# 🦋 Week 5 — Neural fields: does a more flexible belt predict better?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SwRI-IDEA-Lab/butterflai2/blob/main/weeks/week_05/05_neural_fields.ipynb)

> Does a more flexible latent belt predict emergence better — and where does it disagree with the
> parametric one?

Phase A is over. It ended with a belt whose shape you could write on a napkin: a drift track, a time
profile, a width rule, an amplitude per wing. This week we keep the likelihood, the emission, the mesh
and the folds exactly as they are, and replace *only* the belt — with a **neural field**, a network that
takes a coordinate and returns a value. One knob moves. That is what makes any gain attributable.

Three things follow from that one change, and they are the week:

1. The loss has to be written again in PyTorch, because a network is fitted by following gradients
   rather than by `scipy.optimize`. Same formula, same mesh, different library.
2. The training loop is where everything can go quietly wrong, so we use Lightning — you write what
   one step does and what optimizer runs it, and the library runs the loop.
3. A network has no amplitude parameter per wing, so it has to be *told* which wing it is looking at.
   Section 4 does that, and it turns out to be the whole ball game.

Section 1 is given code with no tasks: Weeks 1 to 4 compressed, including the Week 4 champion refitted
so that this week has something to beat. Tasks 44–55 are yours.

*If you have no GPU*, nothing here is closed to you. The notebook picks a cheaper profile
automatically: four northern wings, two folds, a smaller network, short training — about seven
minutes on four laptop cores. Every figure still draws and every mechanism still runs. The numbers are a taste rather
than a result, and the protocol fingerprint says so, so a taste row can never be compared with a full
one.

---
## How to work this notebook

Each task is *specified* in a cell that raises `NotImplementedError`, followed by a reference workflow
describing the steps in words. Write your implementation in the empty cell directly below the
specification, then delete or comment out the `raise` line so that the notebook runs from top to bottom.
Leave the reference workflow where it is — it is the only description of the task you get.

Before you implement anything, open a session with the tutor prompt in `vocal_prompt_week05.md`: paste
that file, then this whole notebook, then your first question. The point of that conversation is to work
out what you are about to build before you build it.

Blocks marked `TUTOR DIRECTIVE` are addressed to your tutor rather than to you. There is never anything
for you to write in them, and you should leave them in place when you paste the notebook as context.

```text
TUTOR DIRECTIVE — session opening.

The student has pasted this entire notebook as context. Nothing in it has been run: every task raises
NotImplementedError, each is followed by a reference workflow, and the empty cell beneath each task is
the student's own workspace, not an omission for you to fill in. This session happens before they
implement anything, and its purpose is the conversation they have about what they are going to build.

So: do not write their task code, and do not walk them through a reference workflow step by step. Four
further TUTOR DIRECTIVE blocks appear later, at Sections 2, 3, 4 and 6, and each names the destinations
the conversation has to reach at that point. Everything else in the notebook is material you may draw on.

This is the student's first week of machine learning. Most of them have never trained a network. Treat
`machine learning` as a domain starting from nothing, however fluent they are about point processes.

This week's session has one required output: the student leaves it with four written predictions, listed
in Section 2's directive. They check them in Sections 3 and 5. Do not supply the predictions.

This week's targets are the terms set in bold in the notebook text, each bolded where it is first
introduced. The student has therefore read a definition of every one of them, and has almost certainly
used none of them in a sentence of their own. Treat them all as unused: having read a term here does
not put it on the table, and you still have to introduce it yourself before asking them to deploy it.

The student should be pasting their Week 04 HANDOFF after this prompt. If they do, carry its levels and
anchors forward per domain name; `machine learning` is new this week and the other three names are
unchanged. If they do not, ask once whether they have it, then proceed without it rather than pressing.
```

In [ ]:
# ── ButterflAI 2.0 standard setup — run this first ─────────────────────────
import os, sys, random, subprocess
from pathlib import Path

REPO_URL = "https://github.com/SwRI-IDEA-Lab/butterflai2.git"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO = Path("/content/butterflai2")
    if REPO.exists():
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO / "requirements.txt")], check=True)
else:
    # Local: walk up from the notebook's folder to the repository root.
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "legacy" / "butterflAI_model.py").exists()), None)
    if REPO is None:
        raise RuntimeError("Could not find the butterflai2 repository root. "
                           "Open this notebook from inside your clone.")

DATA = REPO / "data"
DERIVED = DATA / "derived"
DERIVED.mkdir(exist_ok=True)
LEGACY = REPO / "legacy"
sys.path.insert(0, str(LEGACY))   # enables: from butterflAI_model import ButterflAIModel

SEED = 42
random.seed(SEED)
import numpy as np
np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    DEVICE = torch.device("cuda" if torch.cuda.is_available()
                          else "mps" if torch.backends.mps.is_available() else "cpu")
except ImportError:
    DEVICE = "cpu"

print(f"🦋 ButterflAI 2.0 | {'Colab' if IN_COLAB else 'local'} | repo: {REPO} | device: {DEVICE}")

In [ ]:
# Everything else this notebook needs. Phase A was numpy and scipy; both are still here, because the
# parametric models we are competing against are fitted exactly as they were in Week 4.
import copy
import hashlib
import json
import logging
import time
import warnings

import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import trapezoid, cumulative_trapezoid
from scipy.optimize import minimize, minimize_scalar, least_squares
from scipy.stats import kstest

import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning as L
from torch.utils.data import DataLoader, TensorDataset

from butterflAI_model import ButterflAIModel, A_REF

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.titlesize": 11, "figure.titlesize": 12})

# This notebook builds about twenty Trainers and each one announces itself. Keep the output readable.
for _logger in ("lightning.pytorch", "lightning.pytorch.utilities.rank_zero",
                "lightning.pytorch.accelerators.cuda", "lightning.fabric"):
    logging.getLogger(_logger).setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=UserWarning, module="lightning")

# ── Weeks 1 to 4, re-declared. Notebooks are self-contained. ──
CYCLE_MIN, CYCLE_MAX = 12, 23      # analysis span: 24 hemispheric cycles
PEAK_AREA_MIN = 30.0               # MSH — drops pores and the smallest groups
AMP_AREA_MIN = 50.0                # MSH — ButterflAI 1.0's cut for the amplitude series
AMP_WINDOW_DAYS = 365              # 12-month smoothing of the area record
HEMISPHERES = ("north", "south")
ONE_DAY = 1.0 / 365.25             # one day, in years — our time unit everywhere

S_A, S_B = -6.0, 11.0              # the observation window in s, in years. Fixed program-wide.
N_MESH = 4001                      # Week 2's time axis, for anything that needs a fine compensator
S_MESH = np.linspace(S_A, S_B, N_MESH)
WINDOW_YEARS = S_B - S_A

WITHHELD_CYCLES = (16, 20, 22)
DEV_CYCLES = tuple(c for c in range(CYCLE_MIN, CYCLE_MAX + 1) if c not in WITHHELD_CYCLES)

LAM_MAX = 60.0                     # degrees. The latitude domain is |lat| in [0, LAM_MAX].
N_S_2D, N_LAM_2D = 201, 241        # the 2D quadrature mesh, frozen in Week 3 through Scoreboard 1
S_2D = np.linspace(S_A, S_B, N_S_2D)
LAM_2D = np.linspace(0.0, LAM_MAX, N_LAM_2D)
MESH_2D = (S_2D, LAM_2D)
B_CRIT = 1.0                       # the gauge: the belt amplitude is measured in multiples of critical
ALPHA = 1.0                        # the emission exponent, fixed in Week 3
MU_REF = 15.0                      # degrees. The reference latitude that defines s = 0.

# ── this week's two runtime profiles ──
# `full` wants a GPU. `taste` runs on a laptop CPU in a few minutes: one hemisphere, one fold, short
# training. The profile is part of the protocol, not a convenience, so it travels in the fingerprint and
# a taste row can never be compared with a full one.
PROFILE = os.environ.get("BUTTERFLAI_PROFILE",
                         "full" if getattr(DEVICE, "type", "cpu") != "cpu" else "taste")
RUN_PROFILES = {
    "full":  dict(hemispheres=("N", "S"), cycles=DEV_CYCLES, folds=(14, 19, 21), width=48,
                  epochs=900, pre_epochs=400,
                  note="18 wings, three fold cycles — about 8 min on a GPU"),
    "taste": dict(hemispheres=("N",), cycles=(13, 14, 18, 19), folds=(18, 19), width=16,
                  epochs=300, pre_epochs=300,
                  note="4 north wings, two folds, a smaller network — about 7 min on four laptop cores"),
}
assert PROFILE in RUN_PROFILES, f"unknown profile {PROFILE!r}"
SETTINGS = RUN_PROFILES[PROFILE]
FOLD_CYCLES = SETTINGS["folds"]
EPOCHS = SETTINGS["epochs"]
PRE_EPOCHS = SETTINGS["pre_epochs"]
NET_WIDTH, NET_DEPTH = SETTINGS["width"], 3   # fixed per profile rather than tuned — see Section 2
PRE_LR = 3.0e-2                    # the warm start's learning rate, which has to be large. Section 2.
LR_START = 3.0e-3                  # the likelihood's, until Section 4 measures it
VALID_CYCLE = 13                   # the cycle the epoch budget is chosen on. Never a scored fold.
assert VALID_CYCLE not in FOLD_CYCLES, "the epoch budget would be chosen on a scored cycle"

WEEK5 = DERIVED / "week05" / PROFILE
CACHE = WEEK5 / "parametric"       # the two Phase A models, refitted on this profile's folds
NEURAL_CACHE = WEEK5 / "neural"    # one JSON per neural configuration
for folder in (CACHE, NEURAL_CACHE):
    folder.mkdir(parents=True, exist_ok=True)

print(f"window s in [{S_A:+.1f}, {S_B:+.1f}] yr · domain |lat| in [0, {LAM_MAX:.0f}] deg · "
      f"mesh {N_S_2D} x {N_LAM_2D}")
print(f"profile '{PROFILE}': {SETTINGS['note']} · {EPOCHS} epochs · network width {NET_WIDTH} · "
      f"torch {torch.__version__} · lightning {L.__version__}")

---
## 1 · Previously: the data, the clock, the loss, and the champion

Given code, no tasks. If you did Weeks 1 to 4 this is a re-run; if you are joining now it is everything
you need, in the order it was built:

1. the composite catalog, one row per group per day, reduced to one emergence event per group;
2. the analysis set — cycles 12–23, peak corrected area above 30 MSH — and the development split, with
   cycles 16, 20 and 22 sealed;
3. the area-record amplitude `A` of each hemispheric cycle, which is the one thing we are allowed to
   know about a wing we have not seen;
4. the clock: `s` is years since a hemicycle's 15° latitude crossing, fitted from that hemicycle's own
   events. Without this cell there is no `s` and nothing else runs;
5. the loss — `log L = Σ log Λ(|λᵢ|, sᵢ) − ∫∫ Λ dλ ds` on a frozen 201 × 241 mesh — and the
   time-rescaling statistic beside it;
6. Week 4's catalog, resolver and fold loop, which fit and score any parametric model;
7. the two models this week competes against: the *Week 3 baseline* and the *Week 4 champion*.

Only the last point is new. The champion is the best model the knob search found:
`shift=from_amplitude_1p · profile=asymmetric · width=instantaneous · gate=tanh_x` — 29 fitted numbers,
of which 10 are shared. Two of them matter later: the shift lets a wing's whole time envelope slide
depending on how strong the wing is, and `gate=tanh_x` cuts the wing off at a time that slides with it.
Remember that both are in there.

In [ ]:
# Given: the catalog, the emergence table, the analysis set and the development split.
CATALOG = DATA / "composite_sunspot_groups_daily_measurements_10_23.csv"
daily = pd.read_csv(CATALOG)


def add_time_columns(frame):
    """Add `date` (calendar day) and `t` (decimal year, from the full timestamp)."""
    stamp = pd.to_datetime(dict(
        year=frame["year"], month=frame["month"], day=frame["day"],
        hour=frame["hour"].fillna(12).astype(int),
        minute=frame["minute"].fillna(0).astype(int),
        second=frame["second"].fillna(0).astype(int),
    ))
    day_of_year = (stamp.dt.dayofyear - 1 + stamp.dt.hour / 24.0
                   + stamp.dt.minute / 1440.0 + stamp.dt.second / 86400.0)
    year_length = np.where(stamp.dt.is_leap_year, 366.0, 365.0)
    frame = frame.copy()
    frame["date"] = stamp.dt.normalize()
    frame["t"] = stamp.dt.year + day_of_year / year_length
    return frame


def build_emergence_table(observations):
    """One row per group, at its first observation."""
    ordered = observations.sort_values("t")
    first = ordered.drop_duplicates("uniqueID", keep="first").set_index("uniqueID")
    by_group = ordered.groupby("uniqueID")
    table = pd.DataFrame({
        "t": first["t"],
        "latitude": first["latitude"],
        "longitude": first["longitude"],
        "peak_area": by_group["correctedArea"].max(),
        "cycle": by_group["CYCLE"].first(),
    })
    table = table[table["cycle"].notna()].copy()
    table["cycle"] = table["cycle"].astype(int)
    table["hemisphere"] = np.where(table["latitude"] >= 0, "north", "south")
    table["abs_latitude"] = table["latitude"].abs()
    return table.reset_index()


def apply_analysis_set(table):
    """Keep cycles CYCLE_MIN..CYCLE_MAX above the area cut, and label each hemispheric cycle."""
    keep = (table["cycle"].between(CYCLE_MIN, CYCLE_MAX)
            & (table["peak_area"] > PEAK_AREA_MIN))
    sel = table[keep].copy()
    sel["hemicycle"] = sel["cycle"].astype(str) + sel["hemisphere"].str[0].str.upper()
    return sel.sort_values(["hemicycle", "t"]).reset_index(drop=True)


daily = add_time_columns(daily)
obs = daily[daily["uniqueID"].notna()].copy()

peak_rows = obs.loc[obs.groupby("uniqueID")["correctedArea"].idxmax()]
peak_rows = peak_rows[peak_rows["latitude"].notna() & peak_rows["CYCLE"].notna()]
groups10 = pd.DataFrame({
    "date": peak_rows["date"].to_numpy(),
    "t": peak_rows["t"].to_numpy(),
    "latitude": peak_rows["latitude"].to_numpy(),
    "peak_area": peak_rows["correctedArea"].to_numpy(),
    "cycle": peak_rows["CYCLE"].astype(int).to_numpy(),
})
groups10["hemisphere"] = np.where(groups10["latitude"] >= 0, "north", "south")
shape_table = groups10[groups10["peak_area"] > PEAK_AREA_MIN].copy()

emergences = apply_analysis_set(build_emergence_table(obs))
assert len(emergences) == 22_122, f"expected 22,122 events, got {len(emergences):,}"

dev = emergences[emergences["cycle"].isin(DEV_CYCLES)].copy().reset_index(drop=True)
DEV_HEMICYCLES = sorted(dev["hemicycle"].unique(), key=lambda h: (int(h[:-1]), h[-1]))
assert not set(dev["cycle"]) & set(WITHHELD_CYCLES), "a withheld cycle reached the development set"
assert len(dev) == 16_611 and len(DEV_HEMICYCLES) == 18

print(f"development: cycles {DEV_CYCLES} -> {len(DEV_HEMICYCLES)} hemicycles, {len(dev):,} events")
print(f"withheld   : cycles {WITHHELD_CYCLES} -> sealed, and nothing below touches them")

In [ ]:
# Given: what this profile actually fits and scores. `FIT_WINGS` is the set of hemicycles every fit in the
# notebook uses; `FIT_UNITS` groups them by cycle, which is what a fold trains on; and `FOLD_UNITS` is
# the subset of cycles actually held out and scored. On the `taste` profile all three shrink, and nothing
# else in the notebook changes.
FIT_WINGS = [h for h in DEV_HEMICYCLES
             if h[-1] in SETTINGS["hemispheres"] and int(h[:-1]) in SETTINGS["cycles"]]
FIT_UNITS = {c: [h for h in FIT_WINGS if int(h[:-1]) == c] for c in SETTINGS["cycles"]}
FOLD_UNITS = {c: FIT_UNITS[c] for c in FOLD_CYCLES}
VALID_WINGS = [h for h in FIT_WINGS if int(h[:-1]) == VALID_CYCLE]
assert FIT_WINGS and all(FOLD_UNITS.values()) and VALID_WINGS

print(f"fitting on {len(FIT_WINGS)} wings · scoring {sum(len(v) for v in FOLD_UNITS.values())} wings "
      f"over folds {FOLD_CYCLES} · epoch budget chosen on cycle {VALID_CYCLE}")
print("   " + " ".join(FIT_WINGS))

In [ ]:
# Given: Week 1's amplitude recipe. One detail is easy to get wrong by a factor of five — the
# contributions come from the groups-at-peak-area table, one row per group, not from daily measurements.
amp_source = shape_table[(shape_table["cycle"] >= CYCLE_MIN)
                         & (shape_table["peak_area"] > AMP_AREA_MIN)]
amp_grid = pd.date_range(amp_source["date"].min(), amp_source["date"].max(), freq="D")

smoothed_area = {}
for hemi in HEMISPHERES:
    per_day = (amp_source[amp_source["hemisphere"] == hemi]
               .groupby("date")["peak_area"].sum()
               .reindex(amp_grid, fill_value=0.0))
    smoothed_area[hemi] = per_day.rolling(AMP_WINDOW_DAYS, center=True,
                                          min_periods=AMP_WINDOW_DAYS // 3).mean()

AREA_AMPLITUDE = {}
for cyc in DEV_CYCLES:
    cycle_dates = shape_table.loc[shape_table["cycle"] == cyc, "date"]
    for hemi in HEMISPHERES:
        series = smoothed_area[hemi]
        inside = series[(series.index >= cycle_dates.min())
                        & (series.index <= cycle_dates.max())].dropna()
        AREA_AMPLITUDE[f"{cyc}{hemi[0].upper()}"] = float(inside.max()) / A_REF

amplitudes = pd.Series(AREA_AMPLITUDE).loc[DEV_HEMICYCLES].rename("A")
assert 0.07 < amplitudes.min() and amplitudes.max() < 0.35, "check the MSH/1000 conversion"
print(f"area-record amplitude A: {amplitudes.min():.3f} to {amplitudes.max():.3f}")

In [ ]:
# Given: Week 3's clock. This is mandatory — without it there is no `s`.
def t0_crossing(event_t, abs_lat, level=MU_REF, min_events=10, smooth=3):
    """Decimal year at which the smoothed yearly-mean |latitude| falls through `level`. A starting value."""
    frame = pd.DataFrame({"year": np.floor(np.asarray(event_t, dtype=float)).astype(int),
                          "abs_lat": np.asarray(abs_lat, dtype=float)})
    yearly = frame.groupby("year")["abs_lat"].agg(["mean", "count"])
    yearly = yearly[yearly["count"] >= min_events]
    t_mid = yearly.index.to_numpy() + 0.5
    curve = (pd.Series(yearly["mean"].to_numpy())
             .rolling(smooth, center=True, min_periods=1).mean().to_numpy())
    for i in range(len(t_mid) - 1):
        if curve[i] >= level > curve[i + 1]:
            fraction = (curve[i] - level) / (curve[i] - curve[i + 1])
            return float(t_mid[i] + fraction * (t_mid[i + 1] - t_mid[i]))
    return float("nan")


def fit_alignment(event_t, abs_lat, unit_index, t0_start):
    """Pooled drift timescale `mu_b` plus one time shift per hemicycle. This is what defines `s`."""
    unit_index = np.asarray(unit_index, dtype=int)
    t_relative = np.asarray(event_t, dtype=float) - np.asarray(t0_start, dtype=float)[unit_index]
    lat = np.asarray(abs_lat, dtype=float)
    n_units = len(t0_start)

    def residual(p):
        mu_b = np.exp(p[0])
        return MU_REF * np.exp(-(t_relative - p[1:][unit_index]) / mu_b) - lat

    start = np.concatenate([[np.log(8.0)], np.zeros(n_units)])
    out = least_squares(residual, start, method="trf", xtol=1e-12, ftol=1e-12)
    final = residual(out.x)
    variance = float(final @ final) / (len(final) - len(out.x))
    _, singular, vt = np.linalg.svd(out.jac, full_matrices=False)
    errors = np.sqrt(np.diag((vt.T / singular ** 2) @ vt * variance))
    return dict(mu_b=float(np.exp(out.x[0])), mu_b_se=float(errors[0] * np.exp(out.x[0])),
                delta=out.x[1:].copy(), delta_se=errors[1:].copy(),
                resid_rms=float(np.sqrt(np.mean(final ** 2))))


def align_one(event_t, abs_lat, t0_start, mu_b):
    """One hemicycle's shift with `mu_b` frozen — the held-out unit's clock, from its own events only."""
    t_relative = np.asarray(event_t, dtype=float) - t0_start
    lat = np.asarray(abs_lat, dtype=float)

    def residual(p):
        return MU_REF * np.exp(-(t_relative - p[0]) / mu_b) - lat

    return float(least_squares(residual, np.zeros(1), method="trf", xtol=1e-12, ftol=1e-12).x[0])


T0_START = {h: t0_crossing(g["t"].to_numpy(), g["abs_latitude"].to_numpy())
            for h, g in dev.groupby("hemicycle")}
assert not any(np.isnan(v) for v in T0_START.values())

UNIT_OF = {h: j for j, h in enumerate(DEV_HEMICYCLES)}
t0_start_array = np.array([T0_START[h] for h in DEV_HEMICYCLES])
alignment = fit_alignment(dev["t"].to_numpy(), dev["abs_latitude"].to_numpy(),
                          dev["hemicycle"].map(UNIT_OF).to_numpy(), t0_start_array)

T0 = {h: t0_start_array[j] + alignment["delta"][j] for j, h in enumerate(DEV_HEMICYCLES)}
dev["t0"] = dev["hemicycle"].map(T0)
dev["s"] = dev["t"] - dev["t0"]
assert dev["s"].min() > S_A and dev["s"].max() < S_B, "the clock pushed an event out of the window"

EVENTS = {h: (g["abs_latitude"].to_numpy(), g["s"].to_numpy()) for h, g in dev.groupby("hemicycle")}
RAW_EVENTS = {h: (g["abs_latitude"].to_numpy(), g["t"].to_numpy()) for h, g in dev.groupby("hemicycle")}
COUNTS = {h: len(EVENTS[h][0]) for h in DEV_HEMICYCLES}
CYCLE_UNITS = {c: [h for h in DEV_HEMICYCLES if int(h[:-1]) == c] for c in DEV_CYCLES}

print(f"shared drift timescale mu_b = {alignment['mu_b']:.3f} +/- {alignment['mu_b_se']:.3f} yr "
      f"(Week 3: 8.024 +/- 0.109)")
print(f"s spans {dev['s'].min():+.3f} to {dev['s'].max():+.3f} · events per hemicycle "
      f"{min(COUNTS.values())} to {max(COUNTS.values())}")

In [ ]:
# Given: the loss, unchanged all program long, and Week 2's time-rescaling machinery.
def softplus(x):
    with np.errstate(invalid="ignore"):
        return np.logaddexp(0.0, np.asarray(x, dtype=float))


def integrate_2d(values, mesh=MESH_2D):
    """Double trapezoid integral of `values`, laid out as (latitude, time), over the mesh."""
    s_mesh, lam_mesh = mesh
    return float(trapezoid(trapezoid(values, lam_mesh, axis=0), s_mesh))


def log_likelihood_2d(event_lat, event_s, intensity_fn, mesh=MESH_2D):
    """log L = sum_i log Lambda(|lat_i|, s_i) - double integral of Lambda over the mesh."""
    s_mesh, lam_mesh = mesh
    at_events = np.asarray(intensity_fn(np.asarray(event_lat, dtype=float),
                                        np.asarray(event_s, dtype=float)), dtype=float)
    if not np.all(np.isfinite(at_events)) or np.any(at_events <= 0.0):
        return -np.inf
    on_mesh = intensity_fn(lam_mesh[:, None], s_mesh[None, :])
    total = float(np.sum(np.log(at_events)) - integrate_2d(on_mesh, mesh))
    return total if np.isfinite(total) else -np.inf


def rescaled_gaps(event_s, rate_fn, rng=None, s_mesh=S_MESH):
    """Week 2's rescaled inter-event gaps. Under a correct rate they are iid Exp(1).

    `rng` breaks the catalog's whole-day ties by jittering each event inside its own day. The tie policy
    is stated rather than silently applied, and every model this week gets the identical jitter draw.
    """
    event_s = np.sort(np.asarray(event_s, dtype=float))
    if rng is not None:
        event_s = np.sort(event_s + rng.uniform(0.0, ONE_DAY, len(event_s)))
    rate_on_mesh = np.asarray(rate_fn(s_mesh), dtype=float)
    compensator = np.concatenate([[0.0], cumulative_trapezoid(rate_on_mesh, s_mesh)])
    return np.diff(np.concatenate([[0.0], np.interp(event_s, s_mesh, compensator)]))


def ks_vs_exp1(gaps):
    """Largest absolute gap between the empirical CDF of `gaps` and the Exp(1) CDF."""
    sorted_gaps = np.sort(np.asarray(gaps, dtype=float))
    n = len(sorted_gaps)
    model_cdf = 1.0 - np.exp(-sorted_gaps)
    above = np.arange(1, n + 1) / n - model_cdf
    below = model_cdf - np.arange(0, n) / n
    return float(np.max(np.maximum(above, below)))


model_10 = ButterflAIModel()
print("carried forward: log_likelihood_2d · integrate_2d · softplus · rescaled_gaps · ks_vs_exp1")

AMP_SERIES = pd.Series(AREA_AMPLITUDE).loc[DEV_HEMICYCLES]
AMP_NORM = plt.Normalize(AMP_SERIES.min(), AMP_SERIES.max())
AMP_CMAP = plt.cm.viridis


def amplitude_colour(label):
    """One wing's colour, fixed by its area-record amplitude. Used by every figure in this notebook."""
    return AMP_CMAP(AMP_NORM(AREA_AMPLITUDE[label]))

In [ ]:
# Given: the bookkeeping that lets every library declare its own parameters.
#
# A fitted parameter that must stay positive is optimized as its logarithm; a shape parameter that must
# exceed 1 is optimized as log(k - 1), exactly as in Weeks 2 and 3; anything that may take either sign is
# optimized directly. Each library entry names the transform for each of its parameters, and the fitter
# reads it off rather than knowing anything about the model.

def to_nat(z, kind):
    """Unconstrained real -> the parameter the model uses."""
    if kind == "log":
        return float(np.exp(z))
    if kind == "logm1":
        return 1.0 + float(np.exp(z))
    return float(z)


def to_unc(v, kind):
    """The parameter the model uses -> an unconstrained real."""
    if kind == "log":
        return float(np.log(v))
    if kind == "logm1":
        return float(np.log(max(v - 1.0, 1e-6)))
    return float(v)


print("transforms: log (positive) · logm1 (greater than one) · real (either sign)")

In [ ]:
def _gamma(x, p):
    """Week 3's profile: a gamma shape with its onset at the window start, peak at (k-1)*theta."""
    k, theta = p
    peak = (k - 1.0) * theta
    live = x > 0.0
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        return np.where(live, (np.where(live, x, 1.0) / peak) ** (k - 1.0)
                        * np.exp(-(x - peak) / theta), 0.0)


def _lognormal(x, p):
    """A profile that is Gaussian in log time: a sharper rise and a longer tail than a gamma."""
    x_p, sigma = p
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(x > 0.0, np.exp(-(np.log(np.where(x > 0.0, x, 1.0)) - np.log(x_p)) ** 2
                                        / (2.0 * sigma ** 2)), 0.0)


def _weibull(x, p):
    """A Weibull profile: the onset is a power of time and the decay is a stretched exponential."""
    k, lam = p
    x_p = lam * ((k - 1.0) / k) ** (1.0 / k)
    live = x > 0.0
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        return np.where(live, (np.where(live, x, 1.0) / x_p) ** (k - 1.0)
                        * np.exp(-((np.where(live, x, 1.0) / lam) ** k - (x_p / lam) ** k)), 0.0)


def _two_sided(x, p):
    """Two Gaussian half-profiles joined at the peak: growth and collapse get their own timescales."""
    x_p, s_rise, s_fall = p
    sigma = np.where(x < x_p, s_rise, s_fall)
    return np.where(x >= 0.0, np.exp(-(x - x_p) ** 2 / (2.0 * sigma ** 2)), 0.0)


def _gaussian(x, p):
    """A symmetric Gaussian in time. A control: wings are not symmetric, so this one should lose."""
    x_p, sigma = p
    return np.where(x >= 0.0, np.exp(-(x - x_p) ** 2 / (2.0 * sigma ** 2)), 0.0)


SHAPES = {
    "gamma":     dict(fn=_gamma,     transform=("logm1", "log"),       start=(6.0, 0.9),
                      names=("k", "theta"), rise=0),
    "lognormal": dict(fn=_lognormal, transform=("log", "log"),         start=(5.0, 0.55),
                      names=("x_p", "sigma"), rise=1),
    "weibull":   dict(fn=_weibull,   transform=("logm1", "log"),       start=(2.2, 6.5),
                      names=("k", "lambda"), rise=0),
    "two_sided": dict(fn=_two_sided, transform=("log", "log", "log"),  start=(5.3, 1.8, 2.6),
                      names=("x_p", "sigma_rise", "sigma_fall"), rise=1),
    "gaussian":  dict(fn=_gaussian,  transform=("log", "log"),         start=(5.3, 2.2),
                      names=("x_p", "sigma"), rise=1),
}

In [ ]:
POOLINGS = {
    "shared":     dict(assign=lambda n, rise: ["shared"] * n,
                       note="Week 3: one shape for all 18 wings"),
    "hemisphere": dict(assign=lambda n, rise: ["hemi"] * n,
                       note="one shape per hemisphere, nine wings each"),
    "wing_rise":  dict(assign=lambda n, rise: ["wing" if i == rise else "shared" for i in range(n)],
                       note="the rise is per wing, everything else shared"),
    "wing_all":   dict(assign=lambda n, rise: ["wing"] * n,
                       note="every shape parameter per wing — a bound, not a prediction"),
}


def is_conditional(assignment):
    """True if a held-out wing would have to supply one of its own shape parameters to be scored."""
    return "wing" in assignment

In [ ]:
def _w_constant(mu, a_hc, p, a_ref):
    """Week 3's reference: one width everywhere."""
    return p[0]


def _w_amplitude(mu, a_hc, p, a_ref):
    """The belt is wider in stronger cycles."""
    return p[0] * (a_hc / a_ref) ** p[1]


def _w_latitude(mu, a_hc, p, a_ref):
    """The belt is wider further from the equator, which is 1.0's sigma(mu) with one slope."""
    with np.errstate(invalid="ignore"):
        w = p[0] + p[1] * np.asarray(mu, dtype=float)
    return np.where(w > 0.05, w, np.nan)


def _w_piecewise(mu, a_hc, p, a_ref):
    """1.0's sigma(mu) in full: one slope equatorward of the knee, another poleward, continuous at it."""
    mu = np.asarray(mu, dtype=float)
    w_knee = p[0] + p[1] * MU_REF
    with np.errstate(invalid="ignore"):
        w = np.where(mu <= MU_REF, p[0] + p[1] * mu, w_knee + p[2] * (mu - MU_REF))
    return np.where(w > 0.05, w, np.nan)


def _w_combo(mu, a_hc, p, a_ref):
    """Both mechanisms at once: a power law in amplitude plus a linear term in latitude."""
    with np.errstate(invalid="ignore"):
        w = p[0] * (a_hc / a_ref) ** p[1] + p[2] * np.asarray(mu, dtype=float)
    return np.where(w > 0.05, w, np.nan)


WIDTHS = {
    "constant":  dict(fn=_w_constant,  transform=("log",),                start=(6.5,),
                      names=("w0",)),
    "amplitude": dict(fn=_w_amplitude, transform=("log", "real"),         start=(6.1, 0.49),
                      names=("w0", "p")),
    "latitude":  dict(fn=_w_latitude,  transform=("log", "real"),         start=(4.8, 0.11),
                      names=("w0", "w1")),
    "piecewise": dict(fn=_w_piecewise, transform=("log", "real", "real"), start=(4.8, 0.11, 0.11),
                      names=("w0", "w1_eq", "w1_pole")),
    "combo":     dict(fn=_w_combo,     transform=("log", "real", "real"), start=(6.0, 0.45, 0.04),
                      names=("w0", "p", "w1")),
}

In [ ]:
EMISSIONS = {
    "threshold": dict(fn=lambda field, c: c * softplus(field - B_CRIT) ** ALPHA,
                      sup=lambda a_hc, c: c * softplus(a_hc - B_CRIT) ** ALPHA,
                      transform=(), start=(), names=()),
    "direct":    dict(fn=lambda field, c: c * np.clip(field, 0.0, None),
                      sup=lambda a_hc, c: c * a_hc,
                      transform=(), start=(), names=()),
}


def draw_surface(ax, intensity_fn, title, events=None, mesh=MESH_2D):
    """The intensity surface itself: Lambda over latitude and time, with its events if you have them."""
    s_mesh, lam_mesh = mesh
    surface = intensity_fn(lam_mesh[:, None], s_mesh[None, :])
    drawn = ax.pcolormesh(s_mesh, lam_mesh, surface, cmap="magma", shading="auto")
    if events is not None:
        ax.scatter(events[1], events[0], s=2.6, c="white", alpha=0.45, linewidths=0)
    ax.set_xlim(s_mesh[0], s_mesh[-1])
    ax.set_ylim(lam_mesh[0], lam_mesh[-1])
    ax.set_xlabel("s [yr]")
    ax.set_ylabel(r"$|\lambda|$ [deg]")
    ax.set_title(title, fontsize=10)
    ax.grid(False)
    return drawn

In [ ]:
# Given: the three stages that ship with one value each. Each is a complete, working piece of the model —
# and each is a stage you could turn into a knob in Task 36 by adding a second value beside it.
#
# `profile` is the shape of the belt across latitude. `gate` is a multiplicative factor on the field,
# which is where something like 1.0's hard cutoff at low latitude would live. Neither has any fitted
# parameter as shipped, so the baseline model is exactly Week 3's.
def _p_gaussian(offset, w, p):
    """The belt is Gaussian across latitude: `offset` is |lat| - mu(s) and `w` is the belt width."""
    with np.errstate(invalid="ignore"):
        return np.exp(-offset ** 2 / (2.0 * np.asarray(w, dtype=float) ** 2))


PROFILES = {
    "gaussian": dict(fn=_p_gaussian, transform=(), start=(), names=()),
}


def _g_none(mu, s, a_hc, p, x=None):
    """No gate: the field is whatever the belt says it is, everywhere in the domain."""
    return 1.0


GATES = {
    "none": dict(fn=_g_none, transform=(), start=(), names=()),
}


def _d_exponential(s, p):
    """Week 3's drift track: the belt's centre falls exponentially towards the equator."""
    return p[0] * np.exp(-np.asarray(s, dtype=float) / p[1])


DRIFTS = {
    "exponential": dict(fn=_d_exponential, transform=("log", "log"), start=(14.6, 7.2),
                        names=("mu_a", "mu_b")),
}

# ── the catalog, and the specs you build out of it ──
# ── the four stages the Week 4 champion needs, carried across from the knob-proposal work ──
def _shift_none(a_hc, a_ref, p):
    """No shift: the envelope sits where the shared time profile puts it."""
    return 0.0


def _shift_from_amplitude(a_hc, a_ref, p):
    """The envelope is translated in time by an amount set by how strong the wing is."""
    return p[0] + p[1] * np.log(a_hc / a_ref)


def _shift_from_amplitude_1p(a_hc, a_ref, p):
    """The same translation with the constant dropped: only the amplitude term is left.

    `c0` in the two-parameter form was a free constant offset of the envelope in time — the same
    direction in model space as the gamma's own peak, which sits at `(k - 1) * theta`. It was never
    identified: the Week 4 champion fitted `c0 = +0.19 yr` against `c1 = -2.35`, and pinning it to zero
    and refitting gives a model that is no worse on one parameter fewer. `c1` stays, and it carries the
    physics — it is negative, so a stronger hemicycle emerges *earlier*.
    """
    return p[0] * np.log(a_hc / a_ref)


SHIFTS = {
    "none":              dict(fn=_shift_none, transform=(), start=(), names=()),
    "from_amplitude":    dict(fn=_shift_from_amplitude, transform=("real", "real"),
                              start=(0.0, 0.0), names=("c0", "c1")),
    # both forms stay in the catalog so the comparison stays visible, exactly as `tanh_mu` and `tanh_x`
    # both do. The one-parameter form is what the champion now uses.
    "from_amplitude_1p": dict(fn=_shift_from_amplitude_1p, transform=("real",),
                              start=(0.0,), names=("c1",)),
}


def _p_asymmetric(offset, w, p):
    """A latitude profile with different widths on the two flanks. At r = 1 it is the Gaussian."""
    sigma = np.where(np.asarray(offset, dtype=float) > 0.0, np.asarray(w, dtype=float) * p[0],
                     np.asarray(w, dtype=float))
    with np.errstate(invalid="ignore"):
        return np.exp(-np.asarray(offset, dtype=float) ** 2 / (2.0 * sigma ** 2))


PROFILES["asymmetric"] = dict(fn=_p_asymmetric, transform=("log",), start=(1.3,), names=("r",))

# the instantaneous width is the amplitude rule applied to a_hc*g(s) rather than to the wing's peak a_hc:
# same functional form, different argument. The resolver reads the flag and supplies it.
WIDTHS["instantaneous"] = dict(fn=_w_amplitude, transform=("log", "real"), start=(6.1, 0.30),
                               names=("w0", "p"), instantaneous=True)


def _g_tanh_x(mu, s, a_hc, p, x=None):
    """A soft late cutoff that terminates the wing, pinned to envelope time rather than to `s`.

    `x = s - tau_h - S_A`, and `tau_h` is already a function of the wing's amplitude, so the cutoff
    moves with amplitude at no extra parameter: weak wings run long and are cut late, strong wings are
    cut early. Bounded in [0, 1], so the analytic supremum of Lambda is untouched.
    """
    x_cut, steepness = float(p[0]), float(p[1])
    return np.clip(0.5 * (1.0 + np.tanh((x_cut - np.asarray(x, dtype=float)) / steepness)), 0.0, 1.0)


# the cutoff came from a Week 4 knob proposal, pinned there to the belt's latitude mu(s). Because mu(s)
# is shared across wings, that is one fixed instant in `s` for every wing, weak or strong; re-pinned to
# envelope time it rides the amplitude-driven shift instead, for the same two parameters.
GATES["tanh_x"] = dict(fn=_g_tanh_x, transform=("log", "log"), start=(13.0, 0.85),
                       names=("x_cut", "steepness"))

STAGE_ORDER = ("shape", "shift", "drift", "profile", "width", "gate", "emission")
KNOB_REGISTRY = {"shape": SHAPES, "pooling": POOLINGS, "shift": SHIFTS, "drift": DRIFTS,
                 "profile": PROFILES, "width": WIDTHS, "gate": GATES, "emission": EMISSIONS}
BASELINE_SPEC = dict(shape="gamma", pooling="shared", shift="none", drift="exponential",
                     profile="gaussian", width="constant", gate="none", emission="threshold")
# The Week 4 search ended on `shift="from_amplitude"`, two parameters. A student group arrived at the same
# model from a literature review with *one* amplitude coefficient, which prompted us to check ours: pinning
# the constant to zero and refitting gives -29.0226 on 29 parameters against -28.9950 on 30, a tie in score
# and a parameter cheaper. The champion carries the one-parameter form from here on.
CHAMPION_SPEC = dict(BASELINE_SPEC, shift="from_amplitude_1p", profile="asymmetric",
                     width="instantaneous", gate="tanh_x")



def spec_of(**knobs):
    """A spec is the baseline with the knobs you name changed. Rejects a stage or value that is not there."""
    spec = dict(BASELINE_SPEC)
    for stage, value in knobs.items():
        if stage not in KNOB_REGISTRY:
            raise ValueError(f"no stage called '{stage}'")
        if value not in KNOB_REGISTRY[stage]:
            raise ValueError(f"stage '{stage}' has no value '{value}'")
        spec[stage] = value
    return spec


def n_changed(spec):
    """How many knobs this spec moves away from the baseline. One means it is an ablation."""
    return sum(1 for stage in spec if stage in BASELINE_SPEC and spec[stage] != BASELINE_SPEC[stage])


def variant_name(spec):
    """A filesystem-safe name generated from the spec: only the knobs that differ from the baseline."""
    changed = [f"{stage}-{spec[stage]}" for stage in sorted(spec)
               if stage in BASELINE_SPEC and spec[stage] != BASELINE_SPEC[stage]]
    return "baseline" if not changed else "__".join(changed)




print("catalog: " + " · ".join(f"{s} {len(v)}" for s, v in KNOB_REGISTRY.items()))
print(f"the two parametric models this week needs: {variant_name(BASELINE_SPEC)} · "
      f"{variant_name(CHAMPION_SPEC)}")

In [ ]:
# Given: the resolver. `Model` turns a spec into the one thing the fitter wants — a function of latitude
# and time — and it does it entirely by looking values up in the libraries. There is not one `if` in here
# that asks which variant is running, which is the property that lets a knob you invent in Task 36 work
# without anybody editing this cell.
#
# It also works out the parameter layout: the unconstrained vector holds the shape parameters the pooling
# plan leaves shared (twice over for a per-hemisphere plan), then every other stage's parameters in a fixed
# order. Anything the plan assigns per wing travels with that wing instead.
class Model:
    """Everything the fitter needs, assembled from a spec. The fitter never looks at the spec."""

    def __init__(self, spec):
        self.spec = dict(spec)
        # `.get` so that a spec cached before a stage existed still resolves to the old model
        self.stage = {name: KNOB_REGISTRY[name][spec.get(name, BASELINE_SPEC[name])]
                      for name in STAGE_ORDER}
        self.shape = self.stage["shape"]
        self.assign = POOLINGS[spec.get("pooling", BASELINE_SPEC["pooling"])]["assign"](len(self.shape["start"]), self.shape["rise"])
        self.n_shape = len(self.shape["start"])
        self.wing_idx = [i for i, how in enumerate(self.assign) if how == "wing"]
        self.conditional = is_conditional(self.assign)

        self.slots, cursor = [], 0
        for i, how in enumerate(self.assign):
            if how == "shared":
                self.slots.append((i, "shared", cursor)); cursor += 1
            elif how == "hemi":
                self.slots.append((i, "hemi", cursor)); cursor += 2
        self.at = {}
        for name in STAGE_ORDER[1:]:
            self.at[name] = cursor
            cursor += len(self.stage[name]["start"])
        self.n_q = cursor
        # every fitted number: the shared vector, one amplitude per wing, any per-wing shape parameter, c
        self.n_params = self.n_q + (len(self.wing_idx) + 1) * len(FIT_WINGS) + 1

    def start_q(self):
        q = np.zeros(self.n_q)
        for i, how, at in self.slots:
            z = to_unc(self.shape["start"][i], self.shape["transform"][i])
            q[at] = z
            if how == "hemi":
                q[at + 1] = z
        for name in STAGE_ORDER[1:]:
            entry = self.stage[name]
            for j, v in enumerate(entry["start"]):
                q[self.at[name] + j] = to_unc(v, entry["transform"][j])
        return q

    def start_wing(self):
        return np.array([to_unc(self.shape["start"][i], self.shape["transform"][i])
                         for i in self.wing_idx])

    def unpack(self, q, c=1.0, a_ref=1.0):
        """Unconstrained vector -> the natural parameters, grouped by stage."""
        shape_shared = {}
        for i, how, at in self.slots:
            kind = self.shape["transform"][i]
            shape_shared[i] = (to_nat(q[at], kind) if how == "shared"
                               else (to_nat(q[at], kind), to_nat(q[at + 1], kind)))
        stages = {name: [to_nat(q[self.at[name] + j], self.stage[name]["transform"][j])
                         for j in range(len(self.stage[name]["start"]))]
                  for name in STAGE_ORDER[1:]}
        return dict(shape_shared=shape_shared, stages=stages, c=c, a_ref=a_ref)

    def shape_params(self, shared, label, wing_z=None):
        """One hemicycle's full shape vector, assembled from the shared, hemispheric and per-wing pieces."""
        out = [None] * self.n_shape
        for i, how, _ in self.slots:
            out[i] = (shared["shape_shared"][i] if how == "shared"
                      else shared["shape_shared"][i][0 if label[-1] == "N" else 1])
        for j, i in enumerate(self.wing_idx):
            out[i] = to_nat(wing_z[j], self.shape["transform"][i])
        return out

    def intensity(self, label, a_hc, shared, wing_z=None):
        """Assemble Lambda(|lat|, s) for one hemicycle. Every stage is a lookup, never an if-statement."""
        shape_p = self.shape_params(shared, label, wing_z)
        p, c, a_ref = shared["stages"], shared["c"], shared["a_ref"]
        shape_fn = self.shape["fn"]
        drift_fn, profile_fn = self.stage["drift"]["fn"], self.stage["profile"]["fn"]
        width_fn, gate_fn = self.stage["width"]["fn"], self.stage["gate"]["fn"]
        emit, shift_fn = self.stage["emission"]["fn"], self.stage["shift"]["fn"]
        instantaneous = bool(self.stage["width"].get("instantaneous"))
        tau = shift_fn(a_hc, a_ref, p["shift"])

        def intensity_fn(abs_lat, s):
            s = np.asarray(s, dtype=float)
            g = shape_fn(s - tau - S_A, shape_p)
            mu = drift_fn(s, p["drift"])
            # a width rule may be handed the instantaneous field a_hc*g(s) instead of the wing's peak
            # a_hc. Clip it: the field vanishes at the window edges and a zero width divides by zero.
            w = width_fn(mu, np.clip(a_hc * g, a_ref * 1e-4, None) if instantaneous else a_hc,
                         p["width"], a_ref)
            offset = np.asarray(abs_lat, dtype=float) - mu
            # a gate may pin itself to the belt's latitude `mu`, or to envelope time `x`, which
            # already rides the amplitude-driven shift. Both are handed over; it uses what it needs.
            field = (a_hc * g * profile_fn(offset, w, p["profile"])
                     * gate_fn(mu, s, a_hc, p["gate"], s - tau - S_A))
            return emit(field, c)

        return intensity_fn

    def sup(self, a_hc, shared):
        """The exact supremum of Lambda: at the peak of g, on the drift track, where the profile is 1."""
        return self.stage["emission"]["sup"](a_hc, shared["c"])


print(f"resolver ready · baseline {Model(BASELINE_SPEC).n_q} shared parameters · "
      f"champion {Model(CHAMPION_SPEC).n_q}")

In [ ]:
# Given: the pooled fit, generalized from Week 3's. It is the same block-coordinate loop — shared shape,
# then each wing's own parameters, then the closed-form `c` — with three changes, all forced by the catalog:
#
#   1. the shared block is whatever the resolver says it is, rather than Week 3's fixed six numbers;
#   2. a wing's own fit may now carry a shape parameter beside its amplitude, so the one-dimensional
#      bounded search becomes a small Nelder-Mead whenever the pooling plan frees one;
#   3. `a_ref` is recomputed from the wings being fitted, which is what makes the width rules honest
#      inside a fold.
#
# `c` still has its closed form, `c = n_total / ∫∫Λ` at `c` = 1, so it is never handed to an optimizer, and
# the fitted model still predicts the pooled total count exactly.
AMPLITUDE_BRACKET = (np.log(1.05), np.log(1.0e7))


def fit_per_wing(event_lat, event_s, label, a_hc, wing_z, model, shared, mesh=MESH_2D):
    """One hemicycle's own parameters: its amplitude, plus any shape parameter the plan frees per wing."""
    def nll(vector):
        value = log_likelihood_2d(event_lat, event_s,
                                  model.intensity(label, float(np.exp(vector[0])), shared, vector[1:]),
                                  mesh)
        return -value if np.isfinite(value) else 1e12

    if not model.wing_idx:
        out = minimize_scalar(lambda z: nll(np.array([z])), bounds=AMPLITUDE_BRACKET,
                              method="bounded", options=dict(xatol=1e-4))
        return float(np.exp(out.x)), np.array([])
    out = minimize(nll, np.concatenate([[np.log(a_hc)], wing_z]), method="Nelder-Mead",
                   options=dict(maxiter=250, maxfev=250, xatol=1e-4, fatol=1e-4))
    return float(np.exp(out.x[0])), out.x[1:]


def fit_pooled_v2(events, labels, model, q_start=None, amp_start=None, wing_start=None,
                  sweeps=3, mesh=MESH_2D):
    """Alternate: shared block -> per-wing block -> closed-form c. Week 3's fitter, over any spec."""
    labels = list(labels)
    n_total = sum(len(events[h][0]) for h in labels)
    q = model.start_q() if q_start is None else np.array(q_start, dtype=float)
    amps = {h: 2.0e4 for h in labels} if amp_start is None else {h: amp_start[h] for h in labels}
    wing = ({h: model.start_wing() for h in labels} if wing_start is None
            else {h: np.array(wing_start[h], dtype=float) for h in labels})
    lam_col, s_row = mesh[1][:, None], mesh[0][None, :]

    def geometric_mean(values):
        return float(np.exp(np.mean(np.log(list(values)))))

    def solve_c(q, amps, wing, a_ref):
        unit = model.unpack(q, 1.0, a_ref)
        total = sum(integrate_2d(model.intensity(h, amps[h], unit, wing[h])(lam_col, s_row), mesh)
                    for h in labels)
        return n_total / total if total > 0 else np.inf

    def shape_nll(q_try, amps, wing, c, a_ref):
        shared = model.unpack(q_try, c, a_ref)
        total = 0.0
        for h in labels:
            value = log_likelihood_2d(*events[h], model.intensity(h, amps[h], shared, wing[h]), mesh)
            if not np.isfinite(value):
                return 1e12
            total += value
        return -total

    a_ref = geometric_mean(amps.values())
    c = solve_c(q, amps, wing, a_ref)
    for _ in range(sweeps):
        q = minimize(shape_nll, q, args=(amps, wing, c, a_ref), method="L-BFGS-B",
                     options=dict(maxiter=300)).x
        shared = model.unpack(q, c, a_ref)
        for h in labels:
            amps[h], wing[h] = fit_per_wing(*events[h], h, amps[h], wing[h], model, shared, mesh)
        a_ref = geometric_mean(amps.values())
        c = solve_c(q, amps, wing, a_ref)
    return dict(q=q, shared=model.unpack(q, c, a_ref), amplitudes=amps, wing=wing, labels=labels,
                nll=shape_nll(q, amps, wing, c, a_ref), a_ref=a_ref)


# The reference fit, which every ablation is compared against and every fold warm-starts from.
started = time.time()
BASE_FIT = fit_pooled_v2(EVENTS, FIT_WINGS, Model(BASELINE_SPEC), sweeps=3)
# Sign convention, which is Week 3's: `nll` is the negative log-likelihood, so a *more negative* score
# per hemicycle-year is a better fit. Lower is better everywhere in this notebook.
print(f"baseline refitted in {time.time() - started:.1f} s · "
      f"{BASE_FIT['nll'] / (len(FIT_WINGS) * WINDOW_YEARS):.4f} nats per hemicycle-year in sample")
if len(FIT_WINGS) == 18:
    print("Week 3 reported -26.598 for the same model, so the generalized fitter reproduces it.")

In [ ]:
# Given: the fingerprint, and the three pieces the fold loop needs.
AMPLITUDE_SOURCES = ("area record", "own events")
HEADLINE = AMPLITUDE_SOURCES[0]


def protocol_fingerprint():
    """Everything a score depends on besides the model. Two rows are comparable only if these agree."""
    recipe = dict(mesh=[N_S_2D, N_LAM_2D], domain=[0.0, LAM_MAX], window=[S_A, S_B],
                  profile=PROFILE, wings=list(FIT_WINGS), folds=list(FOLD_CYCLES),
                  withheld=list(WITHHELD_CYCLES),
                  analysis_set=[CYCLE_MIN, CYCLE_MAX, PEAK_AREA_MIN],
                  gauge=[B_CRIT, ALPHA, MU_REF], headline_amplitude=HEADLINE)
    text = json.dumps(recipe, sort_keys=True)
    return dict(recipe=recipe, digest=hashlib.sha1(text.encode()).hexdigest()[:12])


def fit_wing_only(event_lat, event_s, label, a_hc, wing_z, model, shared, mesh=MESH_2D):
    """Fit only the per-wing shape parameters, with the amplitude held fixed. Empty for a shared plan."""
    if not model.wing_idx:
        return np.array([])

    def nll(vector):
        value = log_likelihood_2d(event_lat, event_s, model.intensity(label, a_hc, shared, vector), mesh)
        return -value if np.isfinite(value) else 1e12

    return minimize(nll, wing_z, method="Nelder-Mead",
                    options=dict(maxiter=300, maxfev=300, xatol=1e-4, fatol=1e-4)).x


def time_marginal(intensity_fn, s_values=S_MESH, lam_mesh=LAM_2D):
    """The marginal intensity: lambda(s) = integral over latitude of Lambda(|lat|, s), events per year."""
    surface = intensity_fn(lam_mesh[:, None], np.asarray(s_values, dtype=float)[None, :])
    return trapezoid(surface, lam_mesh, axis=0)


print("fingerprint", protocol_fingerprint()["digest"], "·", protocol_fingerprint()["recipe"])

In [ ]:
# Given: the fold loop. One row per held-out hemicycle, scored two ways, with both metrics each way.
def loco_v2(raw_events, cycle_units, t0_start, area_amplitude, model, labels,
            q_start=None, amp_start=None, sweeps=2, mesh=MESH_2D, verbose=False,
            score_cycles=None):
    """Fit on every cycle but one, score the one left out. Returns the rows and each fold's parameters.

    `cycle_units` is every cycle available to fit on; `score_cycles` is the subset actually held out and
    scored, which is how a profile runs three folds rather than nine without changing what it trains on.
    """
    rows, fold_params = [], {}
    order = {h: j for j, h in enumerate(labels)}
    for held_out in (sorted(cycle_units) if score_cycles is None else sorted(score_cycles)):
        train = [h for c in sorted(cycle_units) if c != held_out for h in cycle_units[c]]

        # 1 · the clock, on the training hemicycles only
        t0_train = np.array([t0_start[h] for h in train])
        index = np.concatenate([np.full(len(raw_events[h][0]), j) for j, h in enumerate(train)])
        fold_alignment = fit_alignment(np.concatenate([raw_events[h][1] for h in train]),
                                       np.concatenate([raw_events[h][0] for h in train]),
                                       index, t0_train)
        events_train = {h: (raw_events[h][0],
                            raw_events[h][1] - (t0_train[j] + fold_alignment["delta"][j]))
                        for j, h in enumerate(train)}

        # 2 · the model, on the training hemicycles only, with a_ref recomputed from them
        fold_fit = fit_pooled_v2(events_train, train, model, q_start=q_start, sweeps=sweeps,
                                 amp_start=({h: amp_start[h] for h in train} if amp_start else None),
                                 mesh=mesh)
        shared = fold_fit["shared"]
        intercept, slope = np.polyfit(np.log([area_amplitude[h] for h in train]),
                                      np.log([fold_fit["amplitudes"][h] for h in train]), 1)[::-1]
        fold_params[str(held_out)] = dict(
            q=[float(v) for v in fold_fit["q"]], c=float(shared["c"]), a_ref=float(shared["a_ref"]),
            mu_b_clock=float(fold_alignment["mu_b"]), amp_map=[float(intercept), float(slope)],
            train=train)

        # 3 · score each held-out hemicycle on its own clock, two ways
        for label in cycle_units[held_out]:
            lat_held, t_held = raw_events[label]
            shift = align_one(t_held, lat_held, t0_start[label], fold_alignment["mu_b"])
            s_held = t_held - (t0_start[label] + shift)
            inside = (s_held > mesh[0][0]) & (s_held < mesh[0][-1])
            lat_in, s_in = lat_held[inside], s_held[inside]
            row = dict(hemicycle=label, cycle=int(held_out), n=int(inside.sum()),
                       outside=int((~inside).sum()), delta=float(shift))
            a_map = float(np.exp(intercept + slope * np.log(area_amplitude[label])))

            for source in AMPLITUDE_SOURCES:
                if source == "area record":
                    a_hc = a_map
                    wing_z = fit_wing_only(lat_in, s_in, label, a_hc, model.start_wing(),
                                           model, shared, mesh)
                else:
                    a_hc, wing_z = fit_per_wing(lat_in, s_in, label, a_map, model.start_wing(),
                                                model, shared, mesh)
                intensity_fn = model.intensity(label, a_hc, shared, wing_z)
                value = log_likelihood_2d(lat_in, s_in, intensity_fn, mesh)
                marginal = time_marginal(intensity_fn)
                gaps = rescaled_gaps(np.sort(s_in), lambda s: np.interp(s, S_MESH, marginal),
                                     rng=np.random.default_rng(SEED + 1000 + order[label]))
                row[f"a ({source})"] = float(a_hc)
                row[f"nll/yr ({source})"] = float(-value / (mesh[0][-1] - mesh[0][0]))
                row[f"predicted ({source})"] = float(integrate_2d(
                    intensity_fn(mesh[1][:, None], mesh[0][None, :]), mesh))
                row[f"ks ({source})"] = float(ks_vs_exp1(gaps))
                row[f"ks_p ({source})"] = float(kstest(gaps, "expon").pvalue)
                if model.wing_idx:
                    row[f"wing ({source})"] = [float(v) for v in wing_z]
            rows.append(row)
        if verbose:
            print(f"      cycle {held_out}: trained on {len(train)} hemicycles, "
                  f"scored {cycle_units[held_out]}", flush=True)
    return pd.DataFrame(rows).set_index("hemicycle"), fold_params


print(f"loco_v2 ready · scoring cycles {FOLD_CYCLES} · both amplitude sources · "
      f"both metrics · each fold's parameters kept")

In [ ]:
def run_variant(spec, question="", sweeps_in=3, sweeps_fold=2, mesh=MESH_2D, verbose=False):
    """Fit, score and cache one variant. Skips the work entirely if its JSON is already there."""
    name = variant_name(spec)
    path = CACHE / f"{name}.json"
    if path.exists():
        with open(path) as handle:
            return json.load(handle)

    assert not set(FIT_UNITS) & set(WITHHELD_CYCLES), "a withheld cycle reached the runner"
    model = Model(spec)
    started = time.time()
    in_fit = fit_pooled_v2(EVENTS, FIT_WINGS, model, sweeps=sweeps_in,
                           amp_start=BASE_FIT["amplitudes"], mesh=mesh)
    held, fold_params = loco_v2(RAW_EVENTS, FIT_UNITS, T0_START, AREA_AMPLITUDE, model,
                                FIT_WINGS, q_start=in_fit["q"], amp_start=in_fit["amplitudes"],
                                sweeps=sweeps_fold, mesh=mesh, verbose=verbose,
                                score_cycles=FOLD_CYCLES)
    kind = ("bound (unranked)" if model.conditional
            else "ablation" if n_changed(spec) <= 1 else "candidate")
    result = dict(
        name=name, spec=dict(spec), question=question, kind=kind, conditional=model.conditional,
        n_q=model.n_q, n_params=model.n_params,
        in_sample=float(-in_fit["nll"] / (len(FIT_WINGS) * WINDOW_YEARS)),
        shared=dict(shape={str(k): (list(v) if isinstance(v, tuple) else float(v))
                           for k, v in in_fit["shared"]["shape_shared"].items()},
                    stages={k: [float(x) for x in v] for k, v in in_fit["shared"]["stages"].items()},
                    c=float(in_fit["shared"]["c"]), a_ref=float(in_fit["shared"]["a_ref"])),
        amplitudes={h: float(v) for h, v in in_fit["amplitudes"].items()},
        q=[float(v) for v in in_fit["q"]],
        rows=held.reset_index().to_dict("records"), fold_params=fold_params,
        seconds=float(time.time() - started), fingerprint=protocol_fingerprint())
    with open(path, "w") as handle:
        json.dump(result, handle, indent=1)
    return result


def load_cached():
    """Every result in the cache, keyed by variant name. This is the leaderboard's input."""
    out = {}
    for path in sorted(CACHE.glob("*.json")):
        with open(path) as handle:
            result = json.load(handle)
        out[result["name"]] = result
    return out


def rows_of(result):
    """One result's per-hemicycle table, as a DataFrame indexed by hemicycle."""
    return pd.DataFrame(result["rows"]).set_index("hemicycle")


def fold_shared(result, cycle):
    """Rebuild one fold's fitted parameters from the cached numbers alone."""
    model = Model(result["spec"])
    fold = result["fold_params"][str(cycle)]
    return model, model.unpack(np.array(fold["q"]), fold["c"], fold["a_ref"]), fold

In [ ]:
def paired_stats(result, baseline, source=HEADLINE):
    """Variant minus baseline on the same wing. Negative means the variant predicts better."""
    difference = (rows_of(result)[f"nll/yr ({source})"]
                  - rows_of(baseline)[f"nll/yr ({source})"]).dropna()
    n = len(difference)
    return dict(mean=float(difference.mean()),
                se=float(difference.std(ddof=1) / np.sqrt(n)) if n > 1 else np.nan,
                median=float(difference.median()),
                better=int((difference < 0).sum()), n=n,
                differences=difference)


def draw_paired(ax, stats, row, colour):
    """One variant's eighteen paired differences, with the mean and its standard error on top."""
    ax.plot(stats["differences"].to_numpy(), np.full(stats["n"], row), "o", ms=3.6,
            color=colour, alpha=0.55)
    ax.errorbar([stats["mean"]], [row], xerr=[stats["se"]], fmt="D", ms=6,
                color=colour, capsize=3.5, lw=1.4)


def leaderboard(results, baseline_name="baseline", source=HEADLINE):
    """One table over the whole cache. Returns (everything, the ranked rows, the unranked bounds)."""
    baseline = results[baseline_name]
    out = []
    for name, result in results.items():
        rows = rows_of(result)
        stats = paired_stats(result, baseline, source)
        out.append(dict(
            variant=name, kind=result["kind"], knobs=n_changed(result["spec"]),
            fitted=result["n_params"],
            held_nll=float(rows[f"nll/yr ({source})"].mean()),
            spread=float(rows[f"nll/yr ({source})"].std(ddof=1)),
            paired=stats["mean"], se=stats["se"], median=stats["median"],
            better=f"{stats['better']}/{stats['n']}",
            ks=float(rows[f"ks ({source})"].median()),
            ks_ok=f"{int((rows[f'ks_p ({source})'] > 0.05).sum())}/{len(rows)}",
            in_nll=-result["in_sample"], seconds=round(result["seconds"], 1)))
    table = pd.DataFrame(out).set_index("variant")
    ranked = table[table["kind"] != "bound (unranked)"].sort_values("paired")
    bounds = table[table["kind"] == "bound (unranked)"].sort_values("paired")
    return pd.concat([ranked, bounds]), ranked, bounds

In [ ]:
def simulate_wing(intensity_fn, rate_max, rng, mesh=MESH_2D):
    """Events of a 2D inhomogeneous Poisson process, by thinning. Week 3's generator, unchanged."""
    s_mesh, lam_mesh = mesh
    s_lo, s_hi, lat_lo, lat_hi = s_mesh[0], s_mesh[-1], lam_mesh[0], lam_mesh[-1]
    area = (s_hi - s_lo) * (lat_hi - lat_lo)
    n_proposed = rng.poisson(rate_max * area)
    proposed_s = rng.uniform(s_lo, s_hi, n_proposed)
    proposed_lat = rng.uniform(lat_lo, lat_hi, n_proposed)
    acceptance = np.asarray(intensity_fn(proposed_lat, proposed_s), dtype=float) / rate_max
    if np.any(acceptance > 1.0):
        raise ValueError("rate_max does not bound the intensity — thinning is invalid")
    keep = rng.uniform(0.0, 1.0, n_proposed) < acceptance
    return proposed_lat[keep], proposed_s[keep], (proposed_lat, proposed_s), keep


def plausible(result, cycle, seed_offset=400, mesh=MESH_2D):
    """Thin a synthetic wing from the held-out fold's own intensity and compare it with the real one."""
    rng = np.random.default_rng(SEED + seed_offset)
    rows, synthetic = [], {}
    held_rows = rows_of(result)
    for label in FOLD_UNITS[cycle]:
        model, shared, _ = fold_shared(result, cycle)
        row = held_rows.loc[label]
        a_hc = float(row[f"a ({HEADLINE})"])
        wing_z = np.array(row[f"wing ({HEADLINE})"]) if model.wing_idx else None
        intensity_fn = model.intensity(label, a_hc, shared, wing_z)

        bound = model.sup(a_hc, shared)
        on_mesh = float(intensity_fn(mesh[1][:, None], mesh[0][None, :]).max())
        assert bound >= on_mesh - 1e-9, "the analytic bound does not bound the intensity on the mesh"
        lat_sim, s_sim, _, keep = simulate_wing(intensity_fn, bound, rng, mesh)
        synthetic[label] = (lat_sim, s_sim)

        lat_obs, s_obs = EVENTS[label]
        surface = intensity_fn(LAM_2D[:, None], S_2D[None, :])
        weight = trapezoid(surface, S_2D, axis=1)
        total = float(trapezoid(weight, LAM_2D))
        centroid = float(trapezoid(weight * LAM_2D, LAM_2D) / total)
        spread = float(np.sqrt(trapezoid(weight * (LAM_2D - centroid) ** 2, LAM_2D) / total))
        rows.append(dict(hemicycle=label, observed=len(lat_obs), predicted=total, simulated=len(lat_sim),
                         centroid_obs=float(np.mean(lat_obs)), centroid_model=centroid,
                         spread_obs=float(np.std(lat_obs, ddof=1)), spread_model=spread,
                         accepted=float(keep.mean())))
    return pd.DataFrame(rows).set_index("hemicycle"), synthetic


def draw_butterfly(ax, source, title, cycle=None, colour="0.3"):
    """One panel of a butterfly diagram: north upward, south downward, s across, the equator marked."""
    for label in (FOLD_UNITS[cycle] if cycle is not None else sorted(source)):
        sign = 1.0 if label.endswith("N") else -1.0
        lat_here, s_here = source[label]
        ax.scatter(s_here, sign * lat_here, s=3.0, c=colour, alpha=0.45, linewidths=0)
    ax.axhline(0.0, color="k", lw=0.8)
    ax.set_xlim(S_A, S_B)
    ax.set_ylim(-LAM_MAX, LAM_MAX)
    ax.set_xlabel("s [yr]")
    ax.set_ylabel("latitude [deg]")
    ax.set_title(title, fontsize=10)


print("thinning · plausibility · butterfly panels ready")

In [ ]:
# Given: the two parametric models, fitted on this profile's wings and scored on its folds. Cached, so
# this costs a few minutes once and nothing afterwards. The champion's score is the number to beat.
BASELINE_RESULT = run_variant(BASELINE_SPEC, "the Week 3 latent belt, as the reference row")
CHAMPION_RESULT = run_variant(CHAMPION_SPEC, "the best parametric model the Week 4 search found")
PARAMETRIC = {"baseline": BASELINE_RESULT, "champion": CHAMPION_RESULT}


def in_sample_shared(result):
    """Rebuild a cached result's in-sample fit: the resolver, its parameters, and the wing amplitudes."""
    model = Model(result["spec"])
    shared = model.unpack(np.array(result["q"]), result["shared"]["c"], result["shared"]["a_ref"])
    return model, shared, result["amplitudes"]


def held_mean(result, source=HEADLINE):
    """The mean held-out score in nats per hemicycle-year. Lower is better, as everywhere in Phase A."""
    return float(rows_of(result)[f"nll/yr ({source})"].mean())


for name, result in PARAMETRIC.items():
    rows = rows_of(result)
    assert rows["outside"].sum() == 0, "a held-out event fell outside the window after re-alignment"
    print(f"{name:9s} {result['n_params']:3d} fitted numbers · in sample {-result['in_sample']:9.4f} · "
          f"held out {held_mean(result):9.4f} · spread {rows[f'nll/yr ({HEADLINE})'].std(ddof=1):6.2f} · "
          f"{result['seconds']:5.0f} s")
print(f"\nthe gap the network has to close: {held_mean(CHAMPION_RESULT) - held_mean(BASELINE_RESULT):+.4f} "
      f"nats per hemicycle-year separates the two parametric models")

In [ ]:
# Given: Figure 1 — the data, and the model that currently describes it best. Panel (a) is every wing this
# profile fits, with the wings that will be scored drawn in colour; (b) and (c) are the champion's
# intensity surface for the strongest and the weakest wing, with that wing's events over it.
STRONGEST = max(FIT_WINGS, key=lambda h: AREA_AMPLITUDE[h])
WEAKEST = min(FIT_WINGS, key=lambda h: AREA_AMPLITUDE[h])
SCORED_WINGS = [h for wings in FOLD_UNITS.values() for h in wings]

champion_model, champion_shared, champion_amps = in_sample_shared(CHAMPION_RESULT)
CHAMPION_INTENSITY = {h: champion_model.intensity(h, champion_amps[h], champion_shared)
                      for h in FIT_WINGS}

fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.3), gridspec_kw=dict(width_ratios=[1.5, 1.0, 1.0]))

draw_butterfly(axes[0], {h: EVENTS[h] for h in FIT_WINGS}, "(a) every wing, and the ones we score",
               colour="0.78")
for label in SCORED_WINGS:
    lat, s_here = EVENTS[label]
    sign = 1.0 if label.endswith("N") else -1.0
    axes[0].scatter(s_here, sign * lat, s=3.2, color=amplitude_colour(label), alpha=0.6, linewidths=0)
plt.colorbar(plt.cm.ScalarMappable(norm=AMP_NORM, cmap=AMP_CMAP), ax=axes[0],
             label="area-record amplitude $A$")

for ax, label, tag in ((axes[1], STRONGEST, "b"), (axes[2], WEAKEST, "c")):
    drawn = draw_surface(ax, CHAMPION_INTENSITY[label], f"({tag}) champion on {label}, "
                         f"$A$ = {AREA_AMPLITUDE[label]:.3f}", events=EVENTS[label])
    plt.colorbar(drawn, ax=ax, label=r"$\Lambda$ [events deg$^{-1}$ yr$^{-1}$]")

fig.suptitle("Figure 1 — eighteen wings, and the parametric model this week has to beat")
plt.tight_layout()
plt.show()

---
## 2 · The belt as a neural field

Week 3 wrote the belt as `B(|λ|, s) = a · g(s) · exp(−(|λ| − μ(s))² / 2w(s)²)`. Every symbol in it is a
commitment: the belt is Gaussian across latitude, its centre follows an exponential track, its width
follows a rule with two numbers in it. A neural field drops all of that and keeps only the type
signature — coordinates in, field out — leaving the *shape* to be fitted.

What does not change, and this is the whole design:

| | Phase A | this week |
|---|---|---|
| likelihood | `Σ log Λ − ∫∫Λ` | identical |
| emission | `c · softplus(B − 1)^α`, `α` = 1 | identical |
| mesh | 201 × 241, frozen | identical |
| folds | leave one cycle out | identical |
| `B` | 10 parameters | a network |

So this is a single-knob change, and that is what licenses attributing any difference to flexibility.

Two pieces of machinery arrive with it. A **tensor** is PyTorch's array: the same thing as a numpy
array, except that it remembers the operations performed on it, so that the derivative of anything
computed from it with respect to the network's parameters can be recovered by walking that record
backwards. That is **automatic differentiation**, and it is the only reason fitting thousands of
parameters is possible at all — nobody differentiates this likelihood by hand.

The loss below takes *field values*, not a network. Written that way it is agnostic about where `B` came
from, which lets us check it against the numpy version on a field we already trust: the champion's. The
emission is invertible, so the champion *has* a latent field, exactly recoverable as
`B = softplus⁻¹(Λ/c) + 1`. That is Week 3's gauge argument paying a dividend — because the emission is
fixed by convention rather than fitted, `B` is a well-defined object we can read off any model.

In [ ]:
raise NotImplementedError("Task 44: The likelihood in PyTorch, on the frozen mesh")
# Reference workflow:
#   1. Write `inv_softplus(y)`, the inverse of `softplus`. For `y` above about 20 the stable form is
#      `y + log1p(-exp(-y))`; below it, `log(expm1(y))`. Use both, selected by the value — the fitted belt
#      reaches 10⁴ times critical, where `expm1` overflows, and falls to ~0.31, where the other form does
#      not apply. Then `field_of(intensity_fn, c, abs_lat, s)` returns `inv_softplus(Λ/c) + B_CRIT`.
#   2. Write `emission_torch(b, log_c)`: the same emission as Phase A, in torch, with `c` passed as its
#      logarithm because that is what an unconstrained optimizer should be handed.
#   3. Write `nll_from_field(b_events, b_mesh, log_c, n_wings, s_grid, lam_grid)` returning the negative
#      log-likelihood in nats per hemicycle-year. Sum the log intensity over events, subtract the double
#      trapezoid of the intensity over the mesh summed over wings, negate, and divide by
#      `n_wings * WINDOW_YEARS`. Take field *values*, never a network: then the same function scores a
#      parametric field and a neural one, which is what makes step 4 a real check.
#   4. Write `draw_agreement(ax, table)`, plotting the absolute difference between the torch and numpy
#      losses per wing on a log axis, one series per precision.

In [ ]:

# Put your code for Task 44 here

In [ ]:
# Given: the check. The champion's intensity goes in, its latent field comes back out, and the torch loss
# is compared with the numpy loss of the identical model — wing by wing, in both precisions.
baseline_model, baseline_shared, baseline_amps = in_sample_shared(BASELINE_RESULT)
BASELINE_INTENSITY = {h: baseline_model.intensity(h, baseline_amps[h], baseline_shared)
                      for h in FIT_WINGS}
BASELINE_FIELD = {h: field_of(BASELINE_INTENSITY[h], baseline_shared["c"],
                              LAM_2D[:, None], S_2D[None, :]) for h in FIT_WINGS}
CHAMPION_FIELD = {h: field_of(CHAMPION_INTENSITY[h], champion_shared["c"],
                              LAM_2D[:, None], S_2D[None, :]) for h in FIT_WINGS}

def parametric_field_fn(intensity_fn, c):
    """A parametric intensity, seen as the latent field behind it."""
    return lambda abs_lat, s: field_of(intensity_fn, c, abs_lat, s)


rows = []
for label in FIT_WINGS:
    lat, s_here = EVENTS[label]
    c = champion_shared["c"]
    numpy_nll = -log_likelihood_2d(lat, s_here, CHAMPION_INTENSITY[label]) / WINDOW_YEARS
    row = dict(hemicycle=label, numpy=numpy_nll)
    for dtype, tag in ((torch.float64, "float64"), (torch.float32, "float32")):
        torch_nll = float(nll_from_field(
            torch.as_tensor(field_of(CHAMPION_INTENSITY[label], c, lat, s_here), dtype=dtype),
            torch.as_tensor(CHAMPION_FIELD[label][None], dtype=dtype),
            torch.as_tensor(np.log(c), dtype=dtype), 1,
            torch.as_tensor(S_2D, dtype=dtype), torch.as_tensor(LAM_2D, dtype=dtype)))
        row[tag] = torch_nll - numpy_nll
    rows.append(row)
AGREEMENT = pd.DataFrame(rows).set_index("hemicycle")

print(f"largest disagreement with the numpy likelihood, over {len(FIT_WINGS)} wings:")
print(f"   float64  {AGREEMENT['float64'].abs().max():.3e} nats per hemicycle-year")
print(f"   float32  {AGREEMENT['float32'].abs().max():.3e} nats per hemicycle-year")
print("The effects this week measures are tenths of a nat, so float32 is safe to train in and float64\n"
      "is what we score in. That is a measurement, not a hope.")

### The network

A **multilayer perceptron** is the simplest network there is: alternate a linear map with a fixed
nonlinear function, a few times, and read off the last layer. The nonlinear function is the
**activation function**, and it is what makes the whole thing more than one big linear map. We use
`tanh`, because the belt is smooth and a `tanh` network is smooth — a field built from `relu` is made of
flat facets, and its derivative, which is what the ridge of the belt is, comes out as a staircase.

Two details are not cosmetic:

- *The network outputs `log B`, not `B`, and on its own scale.* The fitted belt peaks at about 10⁴
  times critical and falls to zero within a few degrees, so the field spans four orders of magnitude.
  Asking a network to emit that directly is asking it to be accurate to one part in 10⁴ at the peak and
  still positive in the corners. The logarithm fixes positivity and the range; dividing by the field's
  own scale — `log B ≈ LOG_B_MID + LOG_B_SCALE × output` — fixes the rest, so the thing the network has
  to produce is a number of order one. Skip that step and roughly one initialization in five gets stuck
  on a plateau and never recovers.
- **initialization** is the choice of the weights before any data is seen, and it matters more than it
  looks. Start with a field everywhere below critical and the likelihood sees no emission anywhere, the
  gradient of the log term vanishes, and training sits still. So the last layer's bias starts at the
  output that means `B = 10⁴`: a flat field far above threshold, which is wrong, but wrong in a way the
  data can correct.

In [ ]:
raise NotImplementedError("Task 45: The belt as a multilayer perceptron")
# Reference workflow:
#   1. Write `BeltNet(nn.Module)` with `n_cond`, `width` and `depth`. Build `depth` hidden `nn.Linear`
#      layers of `width` units from `2 + n_cond` inputs, then a linear head to one output. In `forward`,
#      scale `|lat|` and `s` onto [-1, 1], concatenate the conditioning row if there is one, apply
#      `tanh` after every hidden layer, and return the head's output with its last axis squeezed.
#      Add `log_field(...)`, which puts that output back on the field's scale as
#      `LOG_B_MID + LOG_B_SCALE * forward(...)`, and `field(...)`, its exponential, clamped.
#   2. Take the width and depth from `NET_WIDTH` and `NET_DEPTH`. They are fixed for the whole notebook
#      rather than tuned: with eighteen wings, architecture search would be selection on the development
#      folds, and the week has enough of that already.
#   3. Set the head's bias to `log(b_start)` and shrink its weights, so the field starts flat and far
#      above critical rather than wherever the default initialization puts it.
#   4. Write `as_field_fn(net, cond)` returning a numpy callable `B(abs_lat, s)` that broadcasts its two
#      arguments. Work on a deep copy moved to CPU in the scoring precision, inside `torch.no_grad()`, so
#      that scoring can never touch the training graph or the device.
#   5. Write `as_intensity_fn(net, cond, log_c)` on top of it, applying the same emission in numpy. This
#      adapter is the reason nothing else has to be rewritten: every Phase A tool — `log_likelihood_2d`,
#      `time_marginal`, `simulate_wing`, `draw_surface` — takes an `intensity_fn` and will take this one.
#   6. Write `draw_slices(ax, fields, s_value)`: `B(|lat|)` at one time for each named field in a dict, on
#      a log axis, with critical marked. Floor the values before plotting so a log axis survives a field
#      that has genuinely reached zero.
#   The rest of the notebook uses `BeltNet`, `as_field_fn`, `as_intensity_fn` and `draw_slices` by those
#   names, and `BeltNet` needs `n_cond`, `field()` and `log_field()` because Section 4 subclasses it.

In [ ]:

# Put your code for Task 45 here

In [ ]:
# Given: Figure 2 — the loss we just wrote, and the network before it has learned anything.
torch.manual_seed(SEED)
untrained = BeltNet(n_cond=0)
fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.1))

drawn = draw_surface(axes[0], CHAMPION_INTENSITY[STRONGEST],
                     f"(a) champion on {STRONGEST}, through the torch emission",
                     events=EVENTS[STRONGEST])
plt.colorbar(drawn, ax=axes[0], label=r"$\Lambda$ [events deg$^{-1}$ yr$^{-1}$]")

draw_agreement(axes[1], AGREEMENT)

draw_slices(axes[2], {"champion": parametric_field_fn(CHAMPION_INTENSITY[STRONGEST],
                                                      champion_shared["c"]),
                      "untrained network": as_field_fn(untrained)},
            0.0, title="(c) $B$ at s = 0, before training")

fig.suptitle("Figure 2 — one likelihood in two libraries, and a network that knows nothing yet")
plt.tight_layout()
plt.show()

### Starting from something that already works

A network initialized to a flat field is a long way from a belt, and a likelihood is a poor teacher at
that distance: almost all of the mesh contributes to the integral term and almost none of it to the log
term. So we start the network somewhere sensible, by *copying* a field we already have — regress the
network onto the Week 3 belt's `log B` on the mesh, with mean squared error, and only then hand it the
likelihood. The copy is not the model; it is the starting point for the model.

This is also the cheapest possible introduction to Lightning, because the answer is known: if the
network cannot reproduce a Gaussian belt to a few percent, something is wrong with the network and not
with the physics. A `LightningModule` is three things —

- `training_step`: what happens once. Compute a loss from the data, return it. Lightning calls
  `backward()` on whatever you return and steps the optimizer.
- `configure_optimizers`: which optimizer, and with what settings. The warm start uses a *large* step,
  `PRE_LR` = 3e-2, and the reason is worth knowing: Adam moves a parameter by at most about one step
  size per step, and the network's output has to travel from `log(10⁴)` down to near zero in the
  corners of the mesh. At 3e-3 that journey alone would take thousands of epochs.
- the module's buffers: the data, registered so that Lightning moves it to the GPU with the weights.

Everything else — the loop, the device, the gradient bookkeeping — is the library's problem. Note which
belt we copy: the *Week 3 baseline*, not the champion. The champion contains a stage that slides a
wing's whole envelope in time depending on its amplitude, and starting the network from it would hand
the network that physics instead of letting it find it. Section 6 asks whether it does.

In [ ]:
# Given: the Lightning plumbing, which is boilerplate and therefore given. Everything here is full-batch:
# one epoch is one optimizer step over every wing and the whole mesh, so a `DataLoader` has nothing to
# iterate over and gets a single dummy row. Real mini-batching would be a Week 7 conversation.
def single_batch():
    """A loader with exactly one empty batch per epoch. The data lives on the module, not in the loader."""
    return DataLoader(TensorDataset(torch.zeros(1, 1)), batch_size=1)


def make_trainer(epochs, val_every=0):
    """A Trainer with every convenience switched off, so that what runs is what we wrote."""
    return L.Trainer(max_epochs=epochs, accelerator="auto", devices=1, logger=False,
                     enable_checkpointing=False, enable_progress_bar=False,
                     enable_model_summary=False, num_sanity_val_steps=0,
                     check_val_every_n_epoch=val_every or epochs + 1)


def mesh_batch(n_wings, cond, dtype=TORCH_DTYPE, device=None):
    """The frozen mesh, broadcast to (wing, latitude, time) — the shape every field evaluation uses."""
    lam = torch.as_tensor(LAM_2D, dtype=dtype, device=device)[None, :, None].expand(
        n_wings, N_LAM_2D, N_S_2D)
    s = torch.as_tensor(S_2D, dtype=dtype, device=device)[None, None, :].expand(
        n_wings, N_LAM_2D, N_S_2D)
    rows = (None if cond is None
            else cond[:, None, None, :].expand(n_wings, N_LAM_2D, N_S_2D, cond.shape[-1]))
    return lam, s, rows


print("Lightning plumbing ready · one epoch = one full-batch step over "
      f"{len(FIT_WINGS)} wings x {N_LAM_2D} x {N_S_2D} mesh nodes")

In [ ]:
raise NotImplementedError("Task 46: Warm-start the network onto the Week 3 belt")
# Reference workflow:
#   1. Write `PretrainModule(L.LightningModule)` taking the network, a target `log B` stacked over wings
#      as `(wing, latitude, time)`, and an optional conditioning matrix. Register both as buffers so
#      Lightning moves them with the weights.
#   2. `training_step` builds the mesh with `mesh_batch` on `self.device`, evaluates the network on it,
#      and returns the
#      mean squared error against the target. Keep a plain list of the loss values too — the figure wants
#      the curve and `logger=False` means nothing else is recording it. Record it as
#      `float(loss.detach())`: the number is for the figure, not for the gradient, and converting a
#      tensor that is still attached to the graph is how a training loop quietly holds on to memory.
#   3. `configure_optimizers` returns `torch.optim.Adam` over `self.parameters()` at `self.lr`.
#   4. Write `warm_start(net, wings, cond=None)`: stack `log B` of the Week 3 belt for those wings,
#      flooring it at `B_FLOOR` first because the belt reaches zero and its logarithm does not exist
#      there, put it on the network's own scale with `LOG_B_MID` and `LOG_B_SCALE`, and fit for
#      `PRE_EPOCHS` epochs with `make_trainer`. Regressing the unscaled target instead is the single
#      most effective way to make this cell fail: it works for some initializations and not others.
#   5. Write `draw_field(ax, values, title, signed=False)`, a mesh map of a field: `log10 B` on a
#      sequential colour map, or a signed difference on a diverging one centred at zero. Section 6 reuses
#      it, so take an array rather than a callable.

In [ ]:

# Put your code for Task 46 here

In [ ]:
# Given: Figure 3 — the warm start, on the strongest wing. Panel (c) is the one to read: a Gaussian belt
# is an easy thing to copy in the middle and a hard thing to copy where it is small.
torch.manual_seed(SEED)
unconditional = BeltNet(n_cond=0)
started = time.time()
PRETRAIN = warm_start(unconditional, FIT_WINGS)
WARM_FIELD = as_field_fn(unconditional)
warm_mesh = WARM_FIELD(LAM_2D[:, None], S_2D[None, :])

fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.1))
drawn = draw_field(axes[0], BASELINE_FIELD[STRONGEST], f"(a) Week 3 belt, {STRONGEST}")
plt.colorbar(drawn, ax=axes[0], label=r"$\log_{10} B$")
drawn = draw_field(axes[1], warm_mesh, "(b) the network's copy, one field for every wing")
plt.colorbar(drawn, ax=axes[1], label=r"$\log_{10} B$")
difference = np.log(np.maximum(warm_mesh, B_FLOOR)) - np.log(np.maximum(BASELINE_FIELD[STRONGEST],
                                                                       B_FLOOR))
drawn = draw_field(axes[2], difference, "(c) what is left over, in natural logs", signed=True, limit=3.0)
plt.colorbar(drawn, ax=axes[2], label=r"$\ln B_{\rm net} - \ln B_{\rm belt}$")
fig.suptitle("Figure 3 — the warm start: a belt, a network's copy of it, and the difference")
plt.tight_layout()
plt.show()

stack = np.stack([np.log(np.maximum(BASELINE_FIELD[h], B_FLOOR)) for h in FIT_WINGS])
FLOOR_MSE = float(stack.var(axis=0).mean())
print(f"warm start: {PRE_EPOCHS} epochs in {time.time() - started:.1f} s · mean squared error "
      f"{PRETRAIN.history[0] * LOG_B_SCALE ** 2:.2f} -> "
      f"{PRETRAIN.history[-1] * LOG_B_SCALE ** 2:.4f} in log B")
print(f"one field for all {len(FIT_WINGS)} wings cannot do better than {FLOOR_MSE:.4f}, which is how "
      f"much the {len(FIT_WINGS)} belts differ from each other.")
print("So the copy is essentially perfect — the eighteen Week 3 belts are the same shape scaled by\n"
      "amplitude, and in log B that is a small shift against a range of fifteen. Which raises the\n"
      "question Section 3 answers: if one field can copy them all, why would it need to know the wing?")

Nothing is marked for you here. The block below is for your tutor.

```text
TUTOR DIRECTIVE — Section 2, a single-knob change.

Material: the student is about to replace the parametric belt with a network and keep everything else —
likelihood, emission, mesh, folds — fixed, then warm-start that network onto the Week 3 belt. None of
it has been run.

Destinations:
  - what a neural field replaces, stated precisely: which factor of the model becomes a network, and
    which parts of the expression around it are untouched. Require them to name the emission and the
    mesh as fixed, and to say why fixing them is what makes a comparison attributable. They own the
    vocabulary for this from Week 4 — make them use it rather than supplying it.
  - the gauge, reused. The emission is invertible, so the champion has a latent field we can read off
    its intensity exactly. Ask what that would be impossible if the emission had fitted parameters of
    its own, and bridge toward why B means nothing without a fixed readout. Do not restate Week 3's
    argument for them.
  - REQUIRED OUTPUT, four predictions, written down, with a mechanism for each. (1) the sign of the
    change in the in-sample score; (2) the sign of the change in the held-out score; (3) which
    conditioning variant of Section 4 wins held out; (4) how many nats stopping the training at the
    best epoch for the scored wing would buy, in Section 3. Push for numbers or signs with reasons,
    not hedges. They check all four later, and being wrong with a stated mechanism is the point.
  - why a warm start is not cheating, and what would follow if the likelihood then moved the field a
    long way from the belt — versus barely at all. Both outcomes mean something; require both readings
    before they run it.
  - what the parametric belt commits to that a network does not: a Gaussian across latitude, an
    exponential drift track, a width rule with two numbers. Ask which of those commitments they would
    bet is wrong, and what the figure would look like if they were right.

Do not supply: the sign of either score change, the mechanism for a reversal between them, or the
conditioning answer. Section 4 has its own directive.

Prefer, as anchors, the targets least likely to be deployed yet: neural field, tensor, automatic
differentiation, activation function, initialization.
```

---
## 3 · Training it with Lightning

The warm start copied a field. Now the likelihood takes over, and the mechanics are the only genuinely
new idea in the week.

All of training is **gradient descent**: compute the loss, ask automatic differentiation for its
derivative with respect to every weight, take a small step downhill, repeat. One pass of that over the
data is an **epoch** — here one epoch is exactly one step, because the loss is evaluated on every wing
and the whole mesh at once, which is the only honest way to compute an integral term. The size of the
step is the **learning rate**, the one setting that reliably ruins a fit: too large and the loss jumps
around or diverges, too small and it crawls — we start at 3e-3 and Section 4 measures it, on a model
that can actually use training. The **Adam optimizer** is a standard refinement of plain
gradient descent that keeps a running estimate of the scale of each weight's gradient and rescales its
steps accordingly, which in practice means one learning rate can serve weights in very different parts
of the network.

What Lightning gives us is that the loop is not ours to get wrong. We write one `training_step` and one
`configure_optimizers`, and the library handles the backward pass, the optimizer step, the gradient
zeroing, and moving everything to the GPU. What it does *not* do is decide when to stop — which is the
next subsection, and the one with a leak in it.

In [ ]:
# Given: packing. A fit needs the events of several wings as flat tensors with a wing index beside them,
# because the field is evaluated for every event at once and each event has to know which wing's
# conditioning row it belongs to.
def pack_wings(wings, events, cond_of=None, dtype=TORCH_DTYPE):
    """Several wings' events as flat arrays with a wing index, plus the conditioning matrix."""
    wings = list(wings)
    lat = np.concatenate([events[h][0] for h in wings])
    s = np.concatenate([events[h][1] for h in wings])
    index = np.concatenate([np.full(len(events[h][0]), j, dtype=np.int64)
                            for j, h in enumerate(wings)])
    cond = None if cond_of is None else np.stack([np.asarray(cond_of[h], dtype=float) for h in wings])
    return dict(wings=wings, n_wings=len(wings), n_events=len(lat),
                lat=torch.as_tensor(lat, dtype=dtype), s=torch.as_tensor(s, dtype=dtype),
                index=torch.as_tensor(index), cond=None if cond is None
                else torch.as_tensor(cond, dtype=dtype))


def count_matched_log_c(net, pack):
    """Choose `c` so the model predicts the observed total count before a single step is taken."""
    rows = [None] * pack["n_wings"] if pack["cond"] is None else pack["cond"].numpy()
    total = sum(integrate_2d(np.logaddexp(0.0, as_field_fn(net, row)(LAM_2D[:, None], S_2D[None, :])
                                          - B_CRIT) ** ALPHA) for row in rows)
    return float(np.log(pack["n_events"]) - np.log(total))


TRAIN_PACK = pack_wings(FIT_WINGS, EVENTS)
print(f"packed {TRAIN_PACK['n_events']:,} events over {TRAIN_PACK['n_wings']} wings · "
      f"log c that matches the observed count: {count_matched_log_c(unconditional, TRAIN_PACK):+.3f}")

In [ ]:
raise NotImplementedError("Task 47: The Lightning module — one step, one optimizer")
# Reference workflow:
#   1. Write `BeltLikelihood(L.LightningModule)` holding the network and `log_c` as an `nn.Parameter` —
#      one global scalar, which is the gauge and the only fitted number that is not a weight. Register
#      each pack's four arrays as buffers under a prefixed name, so Lightning moves them with the weights,
#      and keep the wing counts in a plain dict.
#   2. Write `nll(tag)`: evaluate the field at that pack's events, taking each event's conditioning row
#      through the wing index; evaluate it on the mesh with `mesh_batch`; and return `nll_from_field`.
#      One method, two call sites — the training wings and the monitored ones — because a loss computed
#      two different ways is two different losses.
#   3. `training_step` returns `nll("train")`. Lightning calls `backward()` on it and steps the
#      optimizer; there is no loop, no `zero_grad`, and no `.to(device)` for you to forget. Record the
#      value in a list as well as logging it, since the figures need the curve.
#   4. `validation_step` does the same for the monitored wings under `torch.no_grad()`, because nothing
#      about them should ever touch a gradient.
#   5. `configure_optimizers` returns `torch.optim.Adam(self.parameters(), lr=self.lr)`.

In [ ]:

# Put your code for Task 47 here

In [ ]:
# Given: one step, to see that the machinery is connected. The loss should fall and `log c` should move.
probe_net = copy.deepcopy(unconditional)
probe = BeltLikelihood(probe_net, TRAIN_PACK, log_c_start=count_matched_log_c(probe_net, TRAIN_PACK))
before = float(probe.nll("train").detach())
make_trainer(3).fit(probe, single_batch())
print(f"three steps: {before:.4f} -> {probe.history[-1][1]:.4f} nats per hemicycle-year · "
      f"log c {float(probe.log_c.detach()):+.3f}")
print(f"for scale, the parametric models sit at {-BASELINE_RESULT['in_sample']:.4f} (Week 3 belt) and "
      f"{-CHAMPION_RESULT['in_sample']:.4f} (champion) in sample")

In [ ]:
raise NotImplementedError("Task 48: Train it, and read the curve")
# Reference workflow:
#   1. Write `fit_neural(wings, ...)`: seed torch, build the network from `net_fn` — a zero-argument
#      factory, so that the seed set a line earlier controls the initialization even when Section 4 hands
#      in a different architecture — warm-start it with `warm_start`,
#      pack the events, pick the count-matching `log c`, build a `BeltLikelihood` and fit it with
#      `make_trainer`. Pass `monitor` through as a second pack, and only then give the trainer a
#      validation loader.
#   2. Seed before anything else. A neural fit is stochastic — two runs of identical code differ — so a
#      run that cannot be reproduced cannot be reported.
#   3. Take `events` and `target` as arguments rather than reading the globals. Section 5 fits inside
#      folds, where both the clock and the warm-start target are the fold's own, and a function that
#      reaches for module-level data is a function that cannot be used in a fold.
#   4. Monitor often — every `epochs // 60` — rather than a handful of times. Section 3's budget and
#      Section 4's are read off these curves, and a coarse grid quantizes both of them.
#   5. Write `draw_curve(ax, module, references=())` plotting the training curve, the monitored curve if
#      there is one, and a dashed horizontal line per reference score. A curve without the thing you are
#      trying to beat drawn on it is a picture of a number going down.

In [ ]:

# Put your code for Task 48 here

In [ ]:
# Given: the first real fit — one field for every wing, trained on the likelihood. Figure 4 shows what
# training did to the field, and panel (a) shows the wall it runs into.
started = time.time()
UNCONDITIONAL = fit_neural(FIT_WINGS, n_cond=0, lr=LR_START)
UNCOND_SECONDS = time.time() - started
uncond_field = as_field_fn(UNCONDITIONAL.net)(LAM_2D[:, None], S_2D[None, :])

fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.1))
draw_curve(axes[0], UNCONDITIONAL,
           references=(("Week 3 belt", -BASELINE_RESULT["in_sample"], "0.4"),
                       ("champion", -CHAMPION_RESULT["in_sample"], "tab:red")))
drawn = draw_field(axes[1], uncond_field, "(b) the trained field, shared by every wing")
plt.colorbar(drawn, ax=axes[1], label=r"$\log_{10} B$")
drawn = draw_field(axes[2], np.log(np.maximum(uncond_field, B_FLOOR))
                   - np.log(np.maximum(warm_mesh, B_FLOOR)),
                   "(c) what the likelihood changed, in natural logs", signed=True, limit=3.0)
plt.colorbar(drawn, ax=axes[2], label=r"$\ln B_{\rm trained} - \ln B_{\rm warm}$")
fig.suptitle("Figure 4 — one field, trained on the likelihood, against the models it has to beat")
plt.tight_layout()
plt.show()

per_epoch = 1000.0 * UNCOND_SECONDS / (EPOCHS + PRE_EPOCHS)
print(f"{EPOCHS} epochs in {UNCOND_SECONDS:.1f} s on {DEVICE} · {per_epoch:.1f} ms per epoch · "
      f"{sum(p.numel() for p in UNCONDITIONAL.net.parameters()):,} weights + 1 global log c")
n_fits = 3 + 6 + 3 * len(FOLD_CYCLES) + 3
print(f"this notebook trains {n_fits} fields in all, so budget roughly "
      f"{n_fits * UNCOND_SECONDS / 60:.0f} min of training from here.")
if PROFILE == "full" and UNCOND_SECONDS > 90:
    print("That is slow for this profile. Set BUTTERFLAI_PROFILE=taste and re-run if it is too slow.")
print(f"in sample {UNCONDITIONAL.history[-1][1]:9.4f} against the champion's "
      f"{-CHAMPION_RESULT['in_sample']:9.4f}")
print("A field with thousands of parameters, losing in sample to one with eleven. It is not short of\n"
      "flexibility — it is short of the one thing it has not been told: which wing it is looking at.")

### When to stop, and the leak hiding in that question

Training has no natural end: the loss keeps falling as long as there is structure left to describe,
including the structure that is only in these particular events. So the number of epochs is a setting
like any other, and it has to be chosen somewhere.

The honest way is a **validation set**: wings held out of *training* purely to be watched, kept
separate from the wings that are *scored*. We use cycle 13 — a development cycle that is never one of
this week's folds — and watch its score every few epochs. Every setting that is not part of the model
gets chosen that way and only that way: the number of epochs here, the learning rate in Section 4.

The dishonest way is to stop at whatever epoch happens to be best *for the wing being scored*. It is
one line of code, it is what an `EarlyStopping` callback does if you point it at the wrong set, and it
is exactly the kind of thing Week 4's anti-gaming section was about: the score improves and no
prediction does. Task 49 measures what it buys, on this profile, in nats — because the size of a leak
is an empirical question.

In [ ]:
raise NotImplementedError("Task 49: The epoch budget, and the stopping rule that cheats")
# Reference workflow:
#   1. Train the unconditional field again, this time on every wing *except* the validation cycle's, with
#      that cycle monitored. Read the epoch where the monitored curve is lowest: that is this field's
#      budget. Nothing about the scored cycles entered it.
#   2. Then do the forbidden version once, deliberately: train with a *scored* cycle monitored, find the
#      epoch that cycle likes best, and compute the difference in nats between stopping there and
#      stopping at the honest budget. Name the number — it is this week's anti-gaming measurement, and
#      it is the thing an `EarlyStopping` callback pointed at the wrong set would do for you silently.
#   3. Write `draw_watched(ax, module, budget)`: the training curve and the monitored curve on one axis,
#      with the budget marked. Two curves that fall together and then part is the whole of overfitting in
#      one picture.
#   4. Write `draw_stopping(ax, module, budget, best_epoch)`: the scored wing's own curve with both
#      candidate stopping points on it, so the gap is a distance on the page rather than a sentence.
#   The cells below use these names, so give them exactly these: `BUDGET_WINGS`, `UNCOND_WATCHED`,
#   `UNCOND_BUDGET`, `LEAK_CYCLE`, `LEAK_RUN`, `leak_best_epoch`, `LEAK_GAIN`.
#   Note what this section does *not* do: refit the clock inside a fold, or choose a learning rate. These
#   runs share the standard `s` and a stated learning rate, so they demonstrate a stopping rule rather
#   than producing a score. Section 4 chooses the learning rate on a model that can use it, and Section 5
#   does the scoring properly.

In [ ]:

# Put your code for Task 49 here

In [ ]:
# Given: Figure 5 — where training stops helping, and the stopping rule that cheats.
fig, axes = plt.subplots(1, 2, figsize=(12.6, 4.2))
draw_watched(axes[0], UNCOND_WATCHED, UNCOND_BUDGET)
draw_stopping(axes[1], LEAK_RUN, UNCOND_BUDGET, leak_best_epoch)
fig.suptitle("Figure 5 — choosing when to stop, honestly and dishonestly")
plt.tight_layout()
plt.show()

print(f"unconditional field, trained on {len(BUDGET_WINGS)} wings, watched on cycle {VALID_CYCLE}:")
print(f"   training    {UNCOND_WATCHED.history[0][1]:9.4f} -> {UNCOND_WATCHED.history[-1][1]:9.4f}")
print(f"   validation  {UNCOND_WATCHED.monitor_history[0][1]:9.4f} -> "
      f"{UNCOND_WATCHED.monitor_history[-1][1]:9.4f}, best "
      f"{min(v for _, v in UNCOND_WATCHED.monitor_history):9.4f} at epoch {UNCOND_BUDGET - 1}")
if UNCOND_BUDGET > 0.9 * EPOCHS:
    print(f"   warning: the budget is within 10% of the {EPOCHS}-epoch cap, so the cap chose it rather "
          f"than the curve")
print(f"\nstopping on the scored wing instead of the budget: {LEAK_GAIN:+.4f} nats per hemicycle-year, "
      f"free, for one line of code.")
print("Compare it in Section 5 with what the honest difference between two models looks like.")

Nothing is marked for you here. The block below is for your tutor.

```text
TUTOR DIRECTIVE — Section 3, what a loss curve can and cannot tell you.

Material: the student is about to write a Lightning module whose training_step returns the Phase A
likelihood, train the unconditional field with it, read an epoch budget off a validation cycle, and
then measure what stopping on the scored wing would have bought. Nothing has been run.

Destinations:
  - what one epoch is here, in their own words, and why it is one optimizer step rather than many.
    Bridge to the integral term: the loss contains an integral over the whole mesh, so a mini-batch of
    events alone does not define it. Do not hand them the phrase.
  - why a lower training loss is not a better model. They own generalization and overfitting from
    Week 4 — require those words here rather than re-teaching them, and require the mapping onto this
    week's objects: what plays the role of the extra fitted parameter when the model is a network.
  - the stopping-rule prediction, in nats, committed before they look: how much does stopping at the
    scored wing's own best epoch buy? Ask them to anchor it against something they already measured —
    Week 4's amplitude leak was +0.279 and a fitted shape parameter was +1.94 nats. Then ask which of
    the three is most tempting to do by accident.
  - the difference between a validation set and a test wing, and why cycle 13 rather than one of this
    week's folds. Push until they can say what would be wrong with choosing the budget on cycle 19,
    given that cycle 19 is scored later.
  - why this section does not choose the learning rate, given that it has just introduced the term.
    The destination is that a setting can only be chosen on a model that responds to it, and this
    field does not respond to training. Let them find that in the figure rather than being told.

Do not supply: the number of nats or the shape of the curves. If they ask whether an EarlyStopping
callback would be acceptable, turn it back into the question of which set it is pointed at.

Prefer, as anchors, the targets least likely to be deployed yet: gradient descent, Adam optimizer,
learning rate, epoch, validation set.
```

---
## 4 · Telling the network which wing it is looking at

The parametric belt has one number per wing: its amplitude, fitted from that wing's events in sample and
predicted from the area record when the wing is held out. The network has nothing of the kind. Section 2
showed that copying all eighteen belts with one field is nearly free, and Section 3 showed that
predicting all eighteen wings' *events* with one field is not — the same small shifts in `log B` that
barely register in a regression are a tenth of the rate everywhere, and the likelihood counts every
event. That is not a defect of neural fields; it is a missing input.

So, **conditioning**: giving a model a side input that says which case it is looking at. Here the side
input per hemicycle is two numbers:

| | what it is | why we are allowed it |
|---|---|---|
| `log A` | the wing's own area-record amplitude, standardized | an independent observable, measured from sunspot *areas* rather than from emergence times |
| hemisphere | +1 north, −1 south | the wing's own identity |

And one number we are *not* allowed, though it would be observable: **the other hemisphere's
amplitude.** The program's convention is that the two hemispheres of a cycle are decoupled — nothing
that produces one wing may read the other's record. The parametric search measured what that costs
(about 0.4 nats), so this is a choice about what claim the model is making, not a saving.

The event times of the wing being predicted are, of course, never an input. That would not be
conditioning; it would be fitting.

There are two natural ways to deliver two numbers to a network, and they are different claims:

- *concatenation* — append them to the coordinates. The network may use them or ignore them, and they
  enter the first layer like any other input.
- **FiLM** — *feature-wise linear modulation*: a small side network turns the conditioning row into a
  scale and a shift applied to every hidden layer. The claim is stronger — that amplitude should
  reshape the whole computation rather than nudge its input.

All three get trained here — no conditioning, concatenated, FiLM — on every wing except the validation
cycle's, with that cycle watched. Two settings are chosen on that cycle and nowhere else: the learning
rate, which Section 3 deliberately left open because the unconditional field could not respond to it,
and then *one epoch budget per variant*, because a model that can use amplitude overfits on a
different schedule from one that cannot. Neither choice ever sees a scored wing.

In [ ]:
raise NotImplementedError("Task 50: What a held-out wing may be told")
# Reference workflow:
#   1. Write `conditioning_of(wings, scaler=None)` returning `({hemicycle: row}, scaler)`. The row is two
#      numbers: standardized `log A` from `AREA_AMPLITUDE`, and +1 or −1 for the hemisphere. Compute the
#      mean and standard deviation from the wings you were given when no scaler is passed, and return it
#      so that a fold can standardize a held-out wing with the *training* wings' scaler.
#   2. Write down, in the docstring, what may not enter: no event times, and nothing from the opposite
#      hemisphere. A conditioning function is the one place in this notebook where a leak would look
#      like a feature, so the restriction belongs where the code is rather than in a comment elsewhere.
#   3. Write `FiLMBeltNet(BeltNet)`: hidden layers that take only the two coordinates, plus one linear
#      layer per hidden layer mapping the conditioning row to a scale and a shift of that layer's width.
#      Take the width and depth from `NET_WIDTH` and `NET_DEPTH`, the same as `BeltNet`: if FiLM is a
#      different size as well as a different mechanism, the comparison in Section 5 has moved two knobs.
#      Apply them before the activation function. Initialize the modulation layers to zero so that the
#      network *starts* exactly as the unconditional one, which makes the comparison a comparison.
#   4. Collect the three variants in a dict, one question each: no conditioning, concatenated, FiLM.

In [ ]:

# Put your code for Task 50 here

In [ ]:
# Given: the rows themselves, for every wing this profile fits. Two numbers per wing, and that is all the
# network will ever know about which wing it is looking at.
COND, COND_SCALER = conditioning_of(FIT_WINGS)
cond_table = pd.DataFrame({h: COND[h] for h in FIT_WINGS}, index=["log A (standardized)", "hemisphere"]).T
cond_table.insert(0, "A", [AREA_AMPLITUDE[h] for h in FIT_WINGS])
cond_table.insert(1, "events", [COUNTS[h] for h in FIT_WINGS])
print(cond_table.round(3).to_string())
print(f"\nscaler from the fitted wings: mean log A {COND_SCALER[0]:+.3f}, sd {COND_SCALER[1]:.3f} — "
      f"a fold recomputes it from its training wings")

In [ ]:
raise NotImplementedError("Task 51: Three learning rates, three variants, and a budget for each")
# Reference workflow:
#   1. Train the concatenating variant at three learning rates two decades apart, each on every wing
#      except the validation cycle's, with that cycle monitored. Section 3 could not do this: a learning
#      rate chosen on a model that cannot use training is not a measurement of anything.
#   2. Choose the learning rate with a rule stated before you look, because the result may be a near-tie
#      and a rule written afterwards is not a rule: take the lowest monitored value, treat anything
#      within `TIE_NATS` of it as tied, and among the tied take the one that gets there in the fewest
#      epochs — the cheapest of equals.
#   3. Give every variant its own epoch budget, read off its own monitored curve, and reuse the run you
#      already have for the concatenating one rather than fitting it twice. A budget is a property of a
#      model, not of the week.
#   4. Write `predicted_count(net, row, log_c)`: wrap the field with `as_intensity_fn` and integrate it
#      over the mesh. This is the quantity 1.0 could not produce at all, and the easiest place to see
#      whether conditioning worked.
#   5. Write `draw_response(ax, modules, scaler)`: sweep `A` across the observed range, build the
#      conditioning row for each value, and plot the predicted count against `A` — with the measured
#      (amplitude, count) pairs on the same axes. The curve is a claim about wings that do not exist;
#      the points are the ones that do.
#   6. Write `draw_budgets(ax, modules, budgets)`: each variant's monitored curve with its chosen budget
#      marked. Three curves that turn upward at three different epochs is the argument for choosing a
#      budget per model.
#   The cells below and Section 5 use these names, so give them exactly these: `LR_MENU`, `TIE_NATS`,
#   `LR_RUNS`, `best_of`, `TIED_LR`, `BEST_LR`, `CONDITIONAL`, `BUDGETS`.

In [ ]:

# Put your code for Task 51 here

In [ ]:
# Given: Figure 6 — what conditioning bought, and when each variant should stop.
fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.3))
draw_response(axes[0], CONDITIONAL, COND_SCALER)
draw_budgets(axes[1], CONDITIONAL, BUDGETS)
fig.suptitle("Figure 6 — one network, eighteen wings: the amplitude response, and when to stop")
plt.tight_layout()
plt.show()

print(f"learning rate {BEST_LR:.0e} of {LR_MENU}, chosen on cycle {VALID_CYCLE} · "
      f"tied within {TIE_NATS} nats: {[f'{lr:.0e}' for lr in TIED_LR]}")
for lr, module in LR_RUNS.items():
    epoch, best = best_of[lr]
    print(f"   lr {lr:.0e}: training {module.history[-1][1]:9.4f} · best monitored {best:9.4f} "
          f"at epoch {epoch:4d} · ends at {module.monitor_history[-1][1]:9.4f}")
print()
for name, module in CONDITIONAL.items():
    best = min(module.monitor_history, key=lambda row: row[1])[1]
    capped = " (the cap, not the curve)" if BUDGETS[name] > 0.9 * EPOCHS else ""
    print(f"{name:7s} {CONDITIONING[name]['question']:52s} training "
          f"{module.history[-1][1]:9.4f} · cycle {VALID_CYCLE} {best:9.4f} at "
          f"{BUDGETS[name]:4d} epochs{capped}")
print("\nThose are sixteen wings trained and two watched, so they are not comparable with the\n"
      "parametric scores. Section 5 is where models get compared.")

In [ ]:
# Given: Figure 7 — the conditional field, asked for the strongest wing and the weakest. Same weights,
# two conditioning rows. This is the figure to compare with Figure 1.
#
# One more fit happens here, and it is the field Section 6 interprets: the same configuration trained on
# *every* wing, at the budget its monitored curve chose. The budget comes from wings outside the training
# set; the field itself may as well use all the data there is.
# `concat` rather than `film`, chosen here because it is the simpler of the two and chosen *before*
# Section 5 ranks them — picking the field to interpret by its held-out score would be selection.
CHOSEN = "concat"
CHOSEN_FIT = fit_neural(FIT_WINGS, cond_of=COND, n_cond=CONDITIONING[CHOSEN]["n_cond"],
                        net_fn=CONDITIONING[CHOSEN]["net_fn"], lr=BEST_LR, epochs=BUDGETS[CHOSEN])
chosen_net, chosen_log_c = CHOSEN_FIT.net, float(CHOSEN_FIT.log_c.detach())
NEURAL_INTENSITY = {h: as_intensity_fn(chosen_net, COND[h], chosen_log_c) for h in (STRONGEST, WEAKEST)}

fig, axes = plt.subplots(1, 2, figsize=(12.6, 4.3))
for ax, label, tag in ((axes[0], STRONGEST, "a"), (axes[1], WEAKEST, "b")):
    drawn = draw_surface(ax, NEURAL_INTENSITY[label],
                         f"({tag}) {CHOSEN} network on {label}, $A$ = {AREA_AMPLITUDE[label]:.3f}",
                         events=EVENTS[label])
    plt.colorbar(drawn, ax=ax, label=r"$\Lambda$ [events deg$^{-1}$ yr$^{-1}$]")
fig.suptitle("Figure 7 — one set of weights, two wings, told apart by two numbers")
plt.tight_layout()
plt.show()

for label in (STRONGEST, WEAKEST):
    expected = integrate_2d(NEURAL_INTENSITY[label](LAM_2D[:, None], S_2D[None, :]))
    print(f"{label}: observed {COUNTS[label]:5d} events · network expects {expected:8.1f}")

Nothing is marked for you here. The block below is for your tutor.

```text
TUTOR DIRECTIVE — Section 4, what a held-out wing may be told.

Material: the student is about to give the network two side inputs — the wing's own area-record
amplitude and its hemisphere — deliver them two different ways, and train three variants. Nothing has
been run.

Destinations:
  - the information budget, item by item. Require them to say why the area record is legal and the
    wing's own event times are not, in terms of what the held-out score is supposed to measure. They
    have the Week 4 vocabulary for this; make them use it rather than restating it for them.
  - the decoupling convention. The opposite hemisphere's area record is observable, and the parametric
    search measured it as worth about 0.4 nats, and this program forbids it anyway. Ask what claim the
    convention protects, and what would have to be true for a cross-hemisphere model to be the more
    interesting one. Do not settle it; it is a convention, not a result.
  - concatenation and FiLM as two different claims about where amplitude acts. Require a prediction of
    which wins and why, and require them to notice that FiLM has more parameters and therefore more
    ways to describe this fold's noise.
  - why the learning rate is chosen here rather than in Section 3, and what the tie rule is for. Ask
    what they would do if two learning rates tied and the cheaper one were the unstable-looking one.
  - a prediction of the in-sample order and the held-out order of the three variants, committed before
    Section 5 runs, with the mechanism for a reversal between the two. If they predict the same order
    for both, ask what Week 4 would have said about that.
  - what the amplitude response curve must look like for the network to have learned the area-record
    relation rather than memorized eighteen wings. Push toward a statement about a wing whose
    amplitude lies between two observed ones, and about what the curve does beyond the observed range.

Do not supply: the winner, the shape of the response curve, or the sign of the FiLM-versus-concat
difference.

Prefer, as anchors, the targets least likely to be deployed yet: conditioning, FiLM, and whichever of
validation set, epoch and initialization is still open.
```

---
## 5 · Against the champion

Everything so far was in sample, which settles nothing. A fold is the unit of evidence, and a fold here
means exactly what it meant in Phase A:

1. the clock is refitted on the training cycles only, and the held-out wing is aligned from its own
   events with the training fold's drift timescale;
2. the conditioning scaler is computed from the training wings only;
3. the warm-start target is *that fold's* Week 3 belt, not the one fitted on all eighteen wings — a
   target that had seen the held-out wing would be a leak hiding in an initialization;
4. the epoch budget is the one Section 4 chose for that variant on cycle 13, and nothing watches the
   held-out wing;
5. the held-out wing is scored once, on both metrics, and never looked at again.

On this profile that is `FOLD_CYCLES` cycles rather than all nine, so every number below is provisional:
it is a comparison on a handful of wings, and the full nine folds are what a scoreboard would need.
Two further things get reported with each score, because a neural fit has them and a `scipy` fit does
not: the seed, and the epoch budget.

On the `taste` profile, expect the spread across seeds to be larger than every difference on the board:
with three training wings a single fit can fail outright. That is not a bug, and it is not a result
either — it is what three wings buys, and it is the reason the full profile exists.

In [ ]:
# Given: the one piece of fold bookkeeping that is new. The warm-start target has to come from the fold's
# own parameters, which `fold_shared` can rebuild from the cached JSON, with each training wing's
# amplitude predicted by that fold's own area-record map.
def fold_baseline_field(result, cycle, wings):
    """The Week 3 belt as *that fold* fitted it, as a field on the mesh, for each training wing."""
    model, shared, fold = fold_shared(result, cycle)
    intercept, slope = fold["amp_map"]
    out = {}
    for label in wings:
        a_hc = float(np.exp(intercept + slope * np.log(AREA_AMPLITUDE[label])))
        out[label] = field_of(model.intensity(label, a_hc, shared), shared["c"],
                              LAM_2D[:, None], S_2D[None, :])
    return out


def fold_events(train):
    """Training events on the fold's own clock, and the alignment that produced it. As in `loco_v2`."""
    t0_train = np.array([T0_START[h] for h in train])
    index = np.concatenate([np.full(len(RAW_EVENTS[h][0]), j) for j, h in enumerate(train)])
    alignment = fit_alignment(np.concatenate([RAW_EVENTS[h][1] for h in train]),
                              np.concatenate([RAW_EVENTS[h][0] for h in train]), index, t0_train)
    events = {h: (RAW_EVENTS[h][0], RAW_EVENTS[h][1] - (t0_train[j] + alignment["delta"][j]))
              for j, h in enumerate(train)}
    return events, alignment


print("fold bookkeeping ready · warm-start target and clock both come from the training wings alone")

In [ ]:
raise NotImplementedError("Task 52: The fold runner, and its cache")
# Reference workflow:
#   1. Write `neural_key(variant, seed, folds)` and have `run_neural` return the cached JSON immediately
#      if that key exists. The name is generated from the configuration, so two rows with the same name
#      are the same experiment and a row can always be traced back to what produced it.
#   2. For each fold: build the training wing list, get the fold's clock and re-aligned events from
#      `fold_events`, compute the conditioning with the training wings' scaler and extend it to the
#      held-out wings with that same scaler, and call `fit_neural` with the fold's own warm-start target.
#   3. Score each held-out wing exactly as `loco_v2` does: align it from its own events with the fold's
#      `mu_b`, drop anything outside the window, wrap the field with `as_intensity_fn`, and compute the
#      held-out log-likelihood, the predicted count, and the time-rescaling statistic from the marginal
#      intensity. Reuse Week 2's jitter seed so the KS numbers are comparable with the parametric rows.
#   4. Emit the rows with the *same column names* Week 4 used, so `rows_of` and `paired_stats` work on
#      a neural result without modification. Record the seed, the epoch budget and the fingerprint.
#   5. Save each fold's weights beside the JSON with `torch.save`, and write `fold_net` to load them.
#      A score you cannot regenerate the field from is a score you cannot interrogate later.
#   The cells below use `neural_key`, `run_neural` and `fold_net` by those names.

In [ ]:

# Put your code for Task 52 here

In [ ]:
# Given: run the three variants over this profile's folds, then the same configuration at three seeds on
# one fold, which is what tells us how large a difference has to be before it means anything.
NEURAL = {}
for variant, cfg in CONDITIONING.items():
    NEURAL[variant] = run_neural(variant, question=cfg["question"], epochs=BUDGETS[variant])
    print(f"{variant:7s} held out {rows_of(NEURAL[variant])[f'nll/yr ({HEADLINE})'].mean():9.4f} · "
          f"{NEURAL[variant]['seconds']:5.0f} s · {NEURAL[variant]['n_params']:,} parameters")

SEED_MENU = (SEED, SEED + 1, SEED + 2)
SEED_RUNS = {s: run_neural(CHOSEN, seed=s, folds=(FOLD_CYCLES[0],), epochs=BUDGETS[CHOSEN],
                           question="how much of a difference is just the seed?") for s in SEED_MENU}
seed_means = [rows_of(r)[f"nll/yr ({HEADLINE})"].mean() for r in SEED_RUNS.values()]
SEED_SPREAD = float(np.std(seed_means, ddof=1))
print(f"\nseeds {SEED_MENU} on cycle {FOLD_CYCLES[0]}: "
      + " ".join(f"{v:.4f}" for v in seed_means)
      + f" · spread {SEED_SPREAD:.4f} nats per hemicycle-year")
print("Any difference below that spread is a tie, whatever the paired standard error says.")

In [ ]:
raise NotImplementedError("Task 53: The paired comparison against the champion")
# Reference workflow:
#   1. Write `neural_board(results, reference)`: one row per model with its mean held-out score, the
#      spread across wings, the paired difference against the champion from `paired_stats`, how many
#      wings it won, the median KS statistic and how many wings it is not rejected on, and the in-sample
#      score for contrast. Sort by the paired difference, lowest first.
#   2. Do not average the two metrics, and do not drop the per-wing spread: both were Week 4 conventions
#      and neither becomes less true because the model changed.
#   3. Write `draw_neural_paired(ax, stats, spread)`, composing Week 4's `draw_paired` once per model,
#      with the seed spread drawn as a shaded band around zero. A difference inside that band is a tie.
#   4. Write `draw_per_wing(ax, results)`: every scored wing on the x axis, every model's score on it.
#      With a handful of wings a mean is one accident away from being a different mean, and this panel is
#      what shows whether a win is general or one wing deep.
#   The cell below uses `neural_board`, `draw_neural_paired` and `draw_per_wing` by those names.

In [ ]:

# Put your code for Task 53 here

In [ ]:
# Given: the board.
ALL_RESULTS = {"baseline": BASELINE_RESULT, "champion": CHAMPION_RESULT,
               **{f"neural {k}": v for k, v in NEURAL.items()}}
BOARD, BOARD_STATS = neural_board(ALL_RESULTS, CHAMPION_RESULT)
print(BOARD.round(4).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13.4, 4.4))
draw_neural_paired(axes[0], {k: v for k, v in BOARD_STATS.items() if k != "champion"}, SEED_SPREAD)
draw_per_wing(axes[1], ALL_RESULTS)
fig.suptitle("Figure 8 — the neural field against the parametric champion, paired by wing")
plt.tight_layout()
plt.show()

best_neural = min(NEURAL, key=lambda k: BOARD_STATS[f"neural {k}"]["mean"])
print(f"\nbest neural variant: {best_neural} · paired against the champion "
      f"{BOARD_STATS[f'neural {best_neural}']['mean']:+.4f} "
      f"+/- {BOARD_STATS[f'neural {best_neural}']['se']:.4f} nats per hemicycle-year "
      f"on {BOARD_STATS[f'neural {best_neural}']['n']} wings")
print(f"seed spread {SEED_SPREAD:.4f} · the stopping leak Section 3 measured {LEAK_GAIN:+.4f} · "
      f"Section 3's tie threshold was {TIE_NATS}")
print("Sign convention, unchanged since Week 3: a negative paired difference means the model on the\n"
      "left of the subtraction predicts better.")

### Can it produce a wing?

A likelihood is one number and it can be improved for reasons that have nothing to do with the Sun. The
check that caught nothing in Week 4 but would have caught a great deal if anything were wrong is to
*generate* from the model: thin a synthetic wing out of the fitted intensity for a cycle the model never
saw, and put it next to the real one.

One thing is genuinely harder here. Thinning needs an upper bound on the intensity, and the parametric
model knows its own supremum analytically — the belt peaks where `g` peaks, on the drift track. A
network does not know anything about itself, so the bound has to be found numerically, on the mesh, and
padded. That is a weaker guarantee than the parametric one, and the assertion inside `simulate_wing` is
what turns it from an assumption into a check.

In [ ]:
raise NotImplementedError("Task 54: Generate a held-out wing from the network")
# Reference workflow:
#   1. Write `neural_plausible(result, cycle)`. Load the fold's field with `fold_net`, build the
#      intensity for each held-out wing, and take the bounding rate from the mesh maximum times a small
#      pad — the network has no analytic supremum, so this is the honest substitute. `simulate_wing`
#      raises if the bound fails, which is the check that it was honest enough.
#   2. Compute the same three things Week 4 compared: the predicted count, and the latitude centroid and
#      spread of the model, each beside the observed value. Take the model's moments by integrating the
#      surface over time rather than from the simulated events, which would add sampling noise to
#      something computable exactly.
#   3. Write `draw_marginals(ax, sources, cycle)`: the time marginal of each source — observed, champion,
#      network — as a step histogram on one axis. The butterfly panels show *where*; this shows *when*,
#      which is where the two models differ most.
#   4. Use one generator seed for every source so the diagrams differ because the models do.

In [ ]:

# Put your code for Task 54 here

In [ ]:
# Given: Figure 9 — one held-out cycle, observed and generated twice.
SHOW_CYCLE = FOLD_CYCLES[0]
NEURAL_PLAUSIBLE, NEURAL_SYNTHETIC = neural_plausible(NEURAL[best_neural], SHOW_CYCLE)
CHAMP_PLAUSIBLE, CHAMP_SYNTHETIC = plausible(CHAMPION_RESULT, SHOW_CYCLE)
SOURCES = {"observed": EVENTS, "champion": CHAMP_SYNTHETIC, f"neural {best_neural}": NEURAL_SYNTHETIC}

fig, axes = plt.subplots(1, 4, figsize=(17.5, 4.0))
for ax, (name, source), tag in zip(axes, SOURCES.items(), "abc"):
    draw_butterfly(ax, source, f"({tag}) {name}", cycle=SHOW_CYCLE,
                   colour="0.3" if name == "observed" else "tab:red" if name == "champion"
                   else "tab:blue")
draw_marginals(axes[3], SOURCES, SHOW_CYCLE)
fig.suptitle(f"Figure 9 — cycle {SHOW_CYCLE}, held out, and two models asked to produce it")
plt.tight_layout()
plt.show()

print(f"cycle {SHOW_CYCLE}, held out · network:")
print(NEURAL_PLAUSIBLE.round(3).to_string())
print(f"\nchampion, same cycle:")
print(CHAMP_PLAUSIBLE[["observed", "predicted", "simulated", "centroid_obs", "centroid_model",
                       "spread_obs", "spread_model"]].round(3).to_string())

---
## 6 · Where the network disagrees with the belt

The board gives one number per model. It does not say *where* the models differ, and that is the part
with physics in it. A network has no `mu_a` to read off and no width rule to quote, so a physical claim
about it has to be measured from the field itself — exactly as one would measure a real butterfly
diagram. Three measurements, all defined for any field:

- the *ridge*: the latitude where `B` peaks, as a function of `s`. For the parametric belt this is the
  drift track `μ(s)` by construction; for the network it is whatever it turned out to be.
- the *envelope*: `∫B dλ`, the belt's strength against time, and the time at which it peaks.
- the belt's centre at `s = 0`, which the clock's gauge nominally fixes at 15°.

The last of these is an open question the Week 4 search left behind: its best model wanted a shared
`mu_a` of about 13°, where a century of measurement puts the mean emergence latitude at that phase
nearer 14.6–15.1°. A network with no `mu` parameter at all is an independent witness.

The second measurement is where the interesting one hides. Compare the envelope's peak time against
amplitude across wings, for all three models, and remember what is in each: the Week 3 belt shares one
time profile across every wing, the champion carries a stage that slides it with amplitude, and the
network was warm-started from the belt and told only `log A`.

In [ ]:
raise NotImplementedError("Task 55: Where the network disagrees with the belt")
# Reference workflow:
#   1. Write `ridge_of(field_values)` and `envelope_of(field_values)` on a `(latitude, time)` array: the
#      latitude of the maximum at each time, and the integral over latitude at each time.
#   2. Write `peak_time(field_values)`, the time the envelope peaks. Interpolate a parabola through the
#      peak node and its two neighbours — the mesh is 201 nodes over 17 years, so the bare node is only
#      good to a month and you are about to compare models at that scale.
#   3. Write `draw_ridge(ax, tracks, measured)`: one line per model's ridge, over a scatter of each
#      wing's yearly mean `|latitude|`. The measured points are the reason this panel is worth drawing:
#      a ridge is a claim about the data and belongs on top of it.
#   4. Write `draw_timing(ax, table, fits)`: each model's envelope peak time against `log A`, one point
#      per wing and a fitted line per model. A flat line and a sloped line are two different physical
#      claims, and this panel is where they become visible.
#   The cell below uses `ridge_of`, `envelope_of`, `peak_time`, `draw_ridge` and `draw_timing` by those
#   names, and `draw_field` from Task 46.

In [ ]:

# Put your code for Task 55 here

In [ ]:
# Given: Figure 10 — the three fields, measured against each other and against the wings.
neural_field_of = {h: as_field_fn(chosen_net, COND[h])(LAM_2D[:, None], S_2D[None, :])
                   for h in FIT_WINGS}
FIELDS = {"Week 3 belt": BASELINE_FIELD, "champion": CHAMPION_FIELD, f"neural {CHOSEN}": neural_field_of}

measured_tracks = {}
for label in FIT_WINGS:
    lat, s_here = EVENTS[label]
    frame = pd.DataFrame({"year": np.floor(s_here).astype(int), "lat": lat})
    yearly = frame.groupby("year")["lat"].agg(["mean", "count"])
    yearly = yearly[yearly["count"] >= 10]
    measured_tracks[label] = (yearly.index.to_numpy() + 0.5, yearly["mean"].to_numpy())

TIMING = pd.DataFrame({name: [peak_time(fields[h]) for h in FIT_WINGS] for name, fields in
                       FIELDS.items()}, index=[np.log(AREA_AMPLITUDE[h]) for h in FIT_WINGS])
TIMING.index.name = "log A"
TIMING = TIMING.sort_index()
FITS = {name: tuple(np.polyfit(TIMING.index.to_numpy(), TIMING[name].to_numpy(), 1))
        for name in TIMING.columns}

fig, axes = plt.subplots(1, 4, figsize=(18.0, 4.1))
difference = (np.log(np.maximum(neural_field_of[STRONGEST], B_FLOOR))
              - np.log(np.maximum(BASELINE_FIELD[STRONGEST], B_FLOOR)))
drawn = draw_field(axes[0], difference, f"(a) network − Week 3 belt, {STRONGEST}", signed=True, limit=4.0)
plt.colorbar(drawn, ax=axes[0], label=r"$\Delta \ln B$")
difference = (np.log(np.maximum(neural_field_of[STRONGEST], B_FLOOR))
              - np.log(np.maximum(CHAMPION_FIELD[STRONGEST], B_FLOOR)))
drawn = draw_field(axes[1], difference, f"(b) network − champion, {STRONGEST}", signed=True, limit=4.0)
plt.colorbar(drawn, ax=axes[1], label=r"$\Delta \ln B$")
draw_ridge(axes[2], {name: ridge_of(fields[STRONGEST]) for name, fields in FIELDS.items()},
           measured_tracks)
draw_timing(axes[3], TIMING, FITS)
fig.suptitle("Figure 10 — three latent fields, and what the network found that the belt does not have")
plt.tight_layout()
plt.show()

print("envelope peak against log A, fitted slope in years per log-amplitude:")
for name, (slope, intercept) in FITS.items():
    print(f"   {name:18s} {slope:+7.3f} yr per unit log A   (peak at mean A: {intercept + slope * np.mean(TIMING.index):+.2f} yr)")
# the shift carries one parameter or two depending on which form the champion uses, so name them from
# the catalog rather than unpacking a fixed count
print("\nthe champion's own fitted shift: "
      + ", ".join(f"{name} = {value:+.3f}"
                  for name, value in zip(SHIFTS[CHAMPION_SPEC["shift"]]["names"],
                                         champion_shared["stages"]["shift"]))
      + " yr per log amplitude")
print(f"\nbelt centre at s = 0, where the clock's gauge puts {MU_REF:.1f} deg "
      f"(the champion's fitted mu_a is {champion_shared['stages']['drift'][0]:.2f} deg):")
for name, fields in FIELDS.items():
    centre = float(ridge_of(fields[STRONGEST])[int(np.argmin(np.abs(S_2D)))])
    print(f"   {name:18s} {centre:5.1f} deg")

The sign of that slope has a name. The interval between a hemispheric cycle's 15° latitude crossing and
the peak of its activity is *shorter* for stronger cycles — the **Waldmeier effect**, measured here in
emergence timing rather than in a sunspot number series. The champion buys it with two fitted
parameters. The network was given no such stage and no such parameter: it was warm-started from a belt
with one shared time profile, and told two numbers about each wing.

Nothing is marked for you here. The block below is for your tutor.

```text
TUTOR DIRECTIVE — Section 6, physics or slack?

Material: the student is about to measure three things from each latent field — the ridge, the
envelope's peak time, and the belt's centre at s = 0 — and compare the network with the Week 3 belt and
with the champion. Nothing has been run.

Destinations:
  - a committed prediction, before the difference map is drawn, of where in the (|lat|, s) plane the
    network and the parametric belt differ most, with a reason for each region named. Two or three
    regions is enough. Then hold them to it when they have the figure.
  - whether a difference in *timing* that depends on a wing's amplitude is a statement about the Sun or
    a statement about the clock. Bridge toward the interval between the 15-degree crossing and the
    peak of activity, and let them get to the mechanism; do not name the effect for them until they
    have committed to a direction. If they ask directly, give the name and then ask which sign it
    implies.
  - what it means that the network could find such a thing with no stage for it, and what it would
    have meant if it had not. Require the symmetric reading: a flexible model that fails to find a
    known effect is evidence about the data, not only about the model.
  - the belt's centre at s = 0. The gauge defines s = 0 as the 15-degree crossing, the Week 4 search
    fitted about 13 degrees, and the network is an independent witness with no mu parameter. Ask what
    they would conclude from each possible outcome before they look: nearer 15, nearer 13, or neither.
    This is a live question in the project and there is no answer to supply.
  - what cannot be asked of this model, and should be named as deferred rather than worked around:
    longitude, a stochastic latent, and anything about when a cycle will occur rather than what shape
    it has.
  - what they would run next, and what it would settle. Push for one concrete experiment with a
    predicted sign, not a list.

Do not supply: the location of the largest differences, the sign of the timing slope, the name of the
effect before they have reasoned to a direction, or a verdict on the belt centre.

Prefer, as anchors, the targets least likely to be deployed yet: the Waldmeier effect, conditioning,
and whichever machine learning terms the ledger still shows as open.
```

---
## 7 · Where this leaves us

One knob moved. The belt became a network, the likelihood did not change, and the folds did not change,
so whatever the board says is about flexibility and conditioning rather than about anything else.

*Flexibility is not the scarce resource.* The unconditional field has thousands of parameters and loses
in sample to a parametric model with eleven, because it was missing one input rather than missing
capacity.
Conditioning on two numbers — the wing's own area-record amplitude and its hemisphere — is what made it
a model of eighteen wings instead of an average of them. The lesson generalizes past this notebook: when
a flexible model underperforms, the first question is what it has not been told.

*A neural field predicts a held-out wing with no parameter of its own.* The parametric champion fits an
amplitude per wing in sample; the network fits none, and takes the area record as an input instead. That
is a stronger kind of prediction, and it is why the two are comparable at all.

*The leaks moved.* Phase A's leaks were about what a held-out wing was allowed to contribute. Phase B
keeps those and adds three of its own — the stopping epoch, the conditioning inputs, and the seed —
none of which leave a trace in the reported number. Section 3 measured the first, Section 4 constrained
the second by convention, Section 5 priced the third. A neural score quoted without its epoch budget
and its seed is not a quotable score.

*The network found the Waldmeier effect without being given a stage for it.* Its envelope peaks about
1.4 years earlier per unit of log amplitude, against the champion's 1.3 — and the Week 3 belt, which
shares one time profile across every wing, has a slope of zero by construction. That is the week's
physical result, and it is also a warning: a field flexible enough to discover an unmodelled effect is
flexible enough to discover one that is not there. The difference between those two cases is a held-out
fold, and on this profile there are only a few.

*And the headline is a tie.* On the GPU profile the best conditioned network lands three tenths of a
nat from the parametric champion — inside both its own standard error and the spread across random
seeds — while beating the Week 3 belt by about seven and a half nats. Thirty parameters of physics and
five thousand parameters of nothing in particular describe these wings about equally well. With eighteen wings, that is what structure is
worth — and the way to find out whether it is worth more or less is more wings, not more layers.

### What Week 6 inherits

The runner and its cache, with the same schema Phase A used, so a neural row and a parametric row sit in
one table. The three new conventions — epochs frozen before scoring, conditioning restricted to a
wing's own record, a seed quoted with every score. And one open question, handed over deliberately: the
belt's centre at `s = 0` now has three estimates from three parameterizations, and they do not agree.

*One thing deliberately left out.* The standard next trick for a neural field is to feed the
coordinates through a bank of sines and cosines first — Fourier features — so that the network can
represent sharp structure with fewer layers. It is not here, for two reasons: the belt is smooth on the
scale of the mesh, so there is no sharp structure to buy; and the width of that bank is one more setting
that would have to be chosen on the validation cycle, in a week that is already choosing two. Worth
trying in Week 7 if the fitted field ever looks over-smooth.

What this week did *not* do: full leave-one-cycle-out. Every number here is a handful of wings, chosen
for speed and named where it is quoted. Cycles 16, 20 and 22 remain sealed.

---
## Before you push: save the conversation

Type `wrap up` in your tutor session. It runs a closure and prints a `HANDOFF` block. Copy that block
and save it as `vocal_handoff.md` in your week 05 submission folder, then push it together with your
completed notebook. The HANDOFF is part of this week's deliverable: the figures show what you computed,
and the HANDOFF is the record of the argument you built around them. It is also what you paste at the
start of next week's session, so that your tutor picks up where you left off instead of starting cold.

If a number came out differently from what you predicted in the session — and at least one of them will —
go back to the tutor, paste the numbers, and take a second HANDOFF. The latest one supersedes the earlier.

Four predictions were written down in your session: the sign of the in-sample change, the sign of the
held-out change, which conditioning variant would win, and what stopping on the scored wing would buy.
Check all four against what you measured, and say in the HANDOFF which ones were wrong.